<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_09_post_tuning/stage_09_post_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_09 - T2 SEQ2ONE - POST TUNING**

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [1]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-25 00:30:07,318 | INFO | Environment initialized


## **2. Acceso a drive**

In [2]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

2026-04-25 00:30:32,399 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


Mounted at /content/drive


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [3]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
  "t2_p40_h30",
  "t2_p40_h60",
  "t2_p50_h30",
]

# Tamaños de ventana
WINDOW_SIZES = [30]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
        "ema_60",
        "roc_60",
        "roc_30",
        "stoch_k_30",
        "mom_5",
        "atr_norm_10",
        "macd"
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-04-25 00:30:33,015 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-25 00:30:33,016 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-25 00:30:33,017 | INFO | Configuración de experimento cargada
2026-04-25 00:30:33,018 | INFO | Targets: ['t2_p40_h30', 't2_p40_h60', 't2_p50_h30']
2026-04-25 00:30:33,020 | INFO | Window sizes: [30]


In [4]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_mnq_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:10]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[30]["t2_p40_h30"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-04-25 00:30:33,032 | INFO | Paths construidos (windows + scaler compartido T2)
2026-04-25 00:30:33,373 | INFO | Windows OK      : 9
2026-04-25 00:30:33,374 | INFO | Windows missing : 0
2026-04-25 00:30:33,375 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl
2026-04-25 00:30:33,376 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [5]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [6]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [7]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_p40_h30'
        - 't2_p40_h60'
        - 't2_p50_h30'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon_str = target.split("_")[-1]   # ej: "h30"
        horizon = int(horizon_str.replace("h", ""))
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }

### **4.4. Creación de bundles T2**

In [8]:
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundles : dict
        bundles[target] -> bundle dict
    """

    bundles = {}

    # --------------------------
    # Construcción
    # --------------------------
    for target in targets:
        bundles[target] = load_windows_and_scaler(
            window_size=window_size,
            target=target,
            windows_paths=windows_paths,
            scaler_path=scaler_path,
        )

    # --------------------------
    # Verificación rápida
    # --------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    for target in targets:
        b = bundles[target]

        print(f"\nTARGET: {target}")
        print("Train :", b["train"]["X"].shape, b["train"]["y"].shape)
        print("Valid :", b["valid"]["X"].shape, b["valid"]["y"].shape)
        print("Test  :", b["test"]["X"].shape,  b["test"]["y"].shape)
        print("Scaler:", type(b["scaler"]).__name__)

    return bundles

In [9]:
bundles_L30 = create_bundles(window_size=30)

2026-04-25 00:30:34,160 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-25 00:30:34,161 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-25 00:30:34,504 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-25 00:30:34,505 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-25 00:30:34,878 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-25 00:30:34,879 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-25 00:30:36,362 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-25 00:30:36,363 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)
2026-04-25 00:30:36,985 | INFO | Loaded: windows_t2_p40_h60_train.npz
2026-04-25 00:30:36,986 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-25 00:30:37,298 | INFO | Loaded: windows_t2_p40_h60_valid.npz
2026-04-25 00:30:37,298 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-25 00:30:37,655 | INFO | Loaded: windows_t2_p4


WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p40_h60
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

TARGET: t2_p50_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler


Como acceder a las ventanas X e y:

```python
bundle_p40_h30 = bundles_L30["t2_p40_h30"]

X_train = bundle_p40_h30["train"]["X"]
y_train = bundle_p40_h30["train"]["y"]

X_valid = bundle_p40_h30["valid"]["X"]
y_valid = bundle_p40_h30["valid"]["y"]

X_test = bundle_p40_h30["test"]["X"]
y_test = bundle_p40_h30["test"]["y"]

scaler = bundle_p40_h30["scaler"]

print("Target :", bundle_p40_h30["target"])
print("Horizon:", bundle_p40_h30["horizon"])
print("Train  :", X_train.shape, y_train.shape)
print("Valid  :", X_valid.shape, y_valid.shape)
print("Test   :", X_test.shape, y_test.shape)
print("Scaler :", type(scaler).__name__)
```




In [10]:
bundles_L30

{'t2_p40_h30': {'window_size': 30,
  'target': 't2_p40_h30',
  'horizon': 30,
  'paths': {'train': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_train.npz',
   'valid': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_valid.npz',
   'test': '/content/drive/MyDrive/neural_profit/data/07_windows/seq2one/L30/windows_t2_p40_h30_test.npz',
   'scaler': '/content/drive/MyDrive/neural_profit/data/06_scaled/scaler_mnq_t2.pkl'},
  'scaler': StandardScaler(),
  'train': {'X': array([[[-0.11680385, -0.06583842, -0.2238865 , ..., -0.04129868,
            -1.507039  , -0.02778307],
           [ 0.0909589 ,  0.05818945, -0.07248875, ...,  0.3872164 ,
            -1.3723946 ,  0.08631181],
           [-0.18229802, -0.13053213, -0.21077001, ..., -0.1151728 ,
            -1.2194692 ,  0.02307655],
           ...,
           [ 0.40151003,  0.28442496,  0.59289074, ...,  0.47497907,
            -1.0690751 , -0.19294474],
         

### **4.5. Preparación de inputs según el tipo de modelo**

In [11]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

Estos sanity checks sirven para verificar, antes de entrenar, que los datos cargados tengan la estructura correcta y no vengan con errores silenciosos.

En concreto, comprueban que:

- X tenga el formato esperado: 3D (n, seq_len, n_features) o 2D (n, d_flat)
- y tenga forma válida para clasificación seq2one
- X e y tengan la misma cantidad de muestras
- no haya NaN ni inf
- las dimensiones sean consistentes entre train, valid y test
- exista más de una clase en y

Nos conviene tenerlos, porque ayudan a detectar errores de shape o de datos antes de llegar al entrenamiento.

In [12]:
from __future__ import annotations

from typing import Any, Optional, Tuple, Dict, Mapping
import numpy as np


# ============================================================
# SANITY CHECKS PARA DATASETS SEQ2ONE (T2)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía
    ni contenga NaN/inf.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(
    y: np.ndarray,
    *,
    name: str = "y",
    allow_seq_inputs_take_last: bool = False,
) -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)
    - (n, seq_len)       si allow_seq_inputs_take_last=True
    - (n, seq_len, 1)    si allow_seq_inputs_take_last=True
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    if allow_seq_inputs_take_last:
        if y.ndim == 2 and y.shape[1] > 1:
            return y[:, -1]

        if y.ndim == 3 and y.shape[2] == 1:
            return y[:, -1, 0]

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1)"
        f"{' o secuencial si allow_seq_inputs_take_last=True' if allow_seq_inputs_take_last else ''}. "
        f"Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )


def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)
      - y secuencial, opcionalmente, tomando el último valor

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: no aplica directamente.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D.
    allow_seq_inputs_take_last:
        Si y viene como secuencia, toma el último valor.
    """
    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    y = _normalize_y_seq2one(
        y,
        name=f"y[{split_name}]",
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
    )

    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info


def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_p40_h30",
      "horizon": 30,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Si no se pasan expected_*, usa TRAIN como referencia.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        allow_seq_inputs_take_last=allow_seq_inputs_take_last,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_bundles_seq2one(
    bundles: Mapping[str, Dict[str, Any]],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para todos los bundles de un diccionario:

    bundles[target] -> bundle
    """
    results = {}

    for target, bundle in bundles.items():
        results[target] = run_sanity_checks_for_bundle_seq2one(
            bundle,
            tag=target,
            verbose=verbose,
        )

    return results

In [13]:
sanity_results = run_sanity_checks_all_bundles_seq2one(
    bundles_L30,
    verbose=True,
)

[sanity_check_seq2one] train_t2_p40_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h30 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h30 | target=t2_p40_h30 | horizon=30
[sanity_check_seq2one] train_t2_p40_h60 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p40_h60 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] test_t2_p40_h60 | X=(6913, 30, 7) | y=(6913,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
OK t2_p40_h60 | target=t2_p40_h60 | horizon=60
[sanity_check_seq2one] train_t2_p50_h30 | X=(32147, 30, 7) | y=(32147,) | mode=3d | n_classes=3 | classes=[-1, 0, 1]
[sanity_check_seq2one] valid_t2_p50_h30 | X=(6882, 30, 7) | y=(6882,) | mode=3d | n_classes=3 | c

## **6. Módulo de métricas T2**

In [14]:
# ================================
# Setup para importar módulos del proyecto
# ================================

import sys
import importlib

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

# asegurar que metrics es paquete
(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

from metrics.classification_probabilities import (
    compute_probabilistic_outputs,
    apply_decision_rule,
)

print("Módulos importados correctamente")

Módulos importados correctamente


In [15]:
# ================================
# Utilidades: outputs -> DataFrame
# ================================

from __future__ import annotations

import pandas as pd
from typing import Any


def classification_metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output de compute_classification_metrics(...)
    en una fila de DataFrame.

    Usa metrics_to_flat_dict(...) para aplanar la salida
    del módulo classification_metrics y luego agrega metadata
    del experimento.
    """
    flat_metrics = metrics_to_flat_dict(metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_metrics,
    }

    return pd.DataFrame([row])


def _flatten_dict(
    d: dict[str, Any],
    *,
    parent_key: str = "",
    sep: str = "_",
) -> dict[str, Any]:
    """
    Aplana un diccionario arbitrario de forma recursiva.

    Ejemplo:
    {"a": {"b": 1}} -> {"a_b": 1}
    """
    items: dict[str, Any] = {}

    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else str(k)

        if isinstance(v, dict):
            items.update(_flatten_dict(v, parent_key=new_key, sep=sep))
        else:
            items[new_key] = v

    return items


def probabilities_metrics_to_df(
    prob_metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
    horizon: int | None = None,
) -> pd.DataFrame:
    """
    Convierte el output del módulo classification_probabilities
    en una fila de DataFrame.

    Como la estructura puede variar según la implementación,
    se aplana recursivamente el diccionario y luego se agrega
    metadata del experimento.
    """
    flat_prob_metrics = _flatten_dict(prob_metrics)

    row = {
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        **flat_prob_metrics,
    }

    return pd.DataFrame([row])


logger.info("Utilidades de exportación a DataFrame cargadas")

2026-04-25 00:30:40,296 | INFO | Utilidades de exportación a DataFrame cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

### **7.1. Persistencia de métricas y probabilidades**

In [16]:
# ================================
# Persistencia de métricas de clasificación
# ================================

from pathlib import Path
import pandas as pd


def load_classification_metrics_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_best_model" / "classification_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen métricas previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_best_model" / "classification_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_metrics_{model_name}_{split}.parquet"
    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path


# ================================
# Persistencia de probabilidades / decisión
# ================================

def load_classification_probabilities_if_exists(
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_best_model" / "classification_probabilities",
) -> pd.DataFrame:
    """
    Carga resultados probabilísticos si el archivo existe.

    El nombre del archivo incluye modelo y split para evitar
    mezclar resultados de distintos experimentos.
    """
    path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"

    if path.exists():
        logger.info(f"Cargando probabilidades desde: {path}")
        return pd.read_parquet(path)

    logger.info(
        f"No existen probabilidades previas para model={model_name} | split={split}"
    )
    return pd.DataFrame()


def save_classification_probabilities(
    df_probabilities: pd.DataFrame,
    *,
    model_name: str,
    split: str = "valid",
    base_dir: Path = DRIVE_DIR / "metrics_best_model" / "classification_probabilities",
) -> Path:
    """
    Guarda un DataFrame de probabilidades / decisión en formato Parquet.
    """
    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_probabilities_{model_name}_{split}.parquet"
    df_probabilities.to_parquet(out_path, index=False)

    logger.info(f"Probabilidades guardadas en: {out_path}")

    return out_path

### **7.2. Persistencia de modelos**

In [17]:
OUTPUT_DIR_BEST_MODEL = Path("/content/drive/MyDrive/neural_profit/best_models")

In [18]:
from pathlib import Path
import json
import joblib
import pandas as pd


def save_model_artifacts(
    *,
    models: dict,
    output_dir,
    experiment_name: str,
    metrics: pd.DataFrame | None = None,
    probabilities: pd.DataFrame | None = None,
    metadata: dict | None = None,
    overwrite: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Guarda artefactos de entrenamiento de forma genérica para cualquier modelo.

    Estructura:
    output_dir/
        experiment_name/
            models/
                *.joblib
            metrics.parquet
            probabilities.parquet
            metadata.json
    """

    if not isinstance(models, dict) or len(models) == 0:
        raise ValueError("`models` debe ser un diccionario no vacío.")

    output_dir = Path(output_dir)
    artifact_dir = output_dir / experiment_name
    models_dir = artifact_dir / "models"

    artifact_dir.mkdir(parents=True, exist_ok=True)
    models_dir.mkdir(parents=True, exist_ok=True)

    saved_paths = {
        "artifact_dir": str(artifact_dir),
        "models": {},
        "metrics": None,
        "probabilities": None,
        "metadata": None,
    }

    # ==================================================
    # 1) Guardar modelos
    # ==================================================
    for model_key, model_obj in models.items():
        safe_name = str(model_key).replace("/", "__").replace("\\", "__").replace(" ", "_")
        model_path = models_dir / f"{safe_name}.joblib"

        if model_path.exists() and not overwrite:
            raise FileExistsError(f"El modelo ya existe y overwrite=False: {model_path}")

        joblib.dump(model_obj, model_path)
        saved_paths["models"][model_key] = str(model_path)

    # ==================================================
    # 2) Guardar métricas
    # ==================================================
    if metrics is not None:
        metrics_path = artifact_dir / "metrics.parquet"
        if metrics_path.exists() and not overwrite:
            raise FileExistsError(f"El archivo ya existe y overwrite=False: {metrics_path}")
        metrics.to_parquet(metrics_path, index=False)
        saved_paths["metrics"] = str(metrics_path)

    # ==================================================
    # 3) Guardar probabilidades
    # ==================================================
    if probabilities is not None:
        probabilities_path = artifact_dir / "probabilities.parquet"
        if probabilities_path.exists() and not overwrite:
            raise FileExistsError(f"El archivo ya existe y overwrite=False: {probabilities_path}")
        probabilities.to_parquet(probabilities_path, index=False)
        saved_paths["probabilities"] = str(probabilities_path)

    # ==================================================
    # 4) Guardar metadata
    # ==================================================
    metadata_to_save = metadata.copy() if metadata is not None else {}
    metadata_to_save["experiment_name"] = experiment_name
    metadata_to_save["n_models"] = len(models)
    metadata_to_save["model_keys"] = list(models.keys())

    metadata_path = artifact_dir / "metadata.json"
    if metadata_path.exists() and not overwrite:
        raise FileExistsError(f"El archivo ya existe y overwrite=False: {metadata_path}")

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata_to_save, f, ensure_ascii=False, indent=2)

    saved_paths["metadata"] = str(metadata_path)

    if verbose:
        print("\n" + "=" * 80)
        print("ARTEFACTOS GUARDADOS")
        print("=" * 80)
        print(f"artifact_dir   = {artifact_dir}")
        print(f"n_models       = {len(models)}")
        print(f"metrics_saved  = {metrics is not None}")
        print(f"prob_saved     = {probabilities is not None}")
        print(f"metadata_saved = True")

    return saved_paths

### **7.3. Verificación de existencia de experimento**

In [19]:
from pathlib import Path


def make_model_key(model_name: str, target: str, window_size: int) -> str:
    """
    Construye la clave estándar de un modelo entrenado.
    """
    return f"{model_name}__{target}__L{int(window_size)}"


def make_expected_model_keys(
    *,
    model_name: str,
    targets: list[str],
    window_size: int,
) -> list[str]:
    """
    Genera las claves esperadas de modelos para un experimento.
    """
    return [
        make_model_key(model_name=model_name, target=target, window_size=window_size)
        for target in targets
    ]


def check_experiment_artifacts_exist(
    *,
    output_dir,
    experiment_name: str,
    expected_model_keys: list[str] | None = None,
    require_metrics: bool = True,
    require_probabilities: bool = True,
) -> dict:
    """
    Verifica si ya existen los artefactos de un experimento.

    Estructura esperada:
    output_dir/
        experiment_name/
            models/
                *.joblib
            metrics.parquet
            probabilities.parquet
            metadata.json
    """

    output_dir = Path(output_dir)
    artifact_dir = output_dir / experiment_name
    models_dir = artifact_dir / "models"

    metrics_path = artifact_dir / "metrics.parquet"
    probabilities_path = artifact_dir / "probabilities.parquet"
    metadata_path = artifact_dir / "metadata.json"

    metrics_exists = metrics_path.exists()
    probabilities_exists = probabilities_path.exists()
    metadata_exists = metadata_path.exists()
    models_dir_exists = models_dir.exists()

    existing_model_files = []
    if models_dir_exists:
        existing_model_files = sorted([p.name for p in models_dir.glob("*.joblib")])

    missing_model_keys = []
    if expected_model_keys is not None:
        for model_key in expected_model_keys:
            safe_name = (
                str(model_key)
                .replace("/", "__")
                .replace("\\", "__")
                .replace(" ", "_")
            )
            model_path = models_dir / f"{safe_name}.joblib"
            if not model_path.exists():
                missing_model_keys.append(model_key)

    models_ok = models_dir_exists
    if expected_model_keys is not None:
        models_ok = models_ok and (len(missing_model_keys) == 0)

    metrics_ok = metrics_exists if require_metrics else True
    probabilities_ok = probabilities_exists if require_probabilities else True

    exists_all = artifact_dir.exists() and models_ok and metrics_ok and probabilities_ok

    return {
        "exists_all": exists_all,
        "artifact_dir": str(artifact_dir),
        "models_dir": str(models_dir),
        "metrics_path": str(metrics_path),
        "probabilities_path": str(probabilities_path),
        "metadata_path": str(metadata_path),
        "models_dir_exists": models_dir_exists,
        "metrics_exists": metrics_exists,
        "probabilities_exists": probabilities_exists,
        "metadata_exists": metadata_exists,
        "existing_model_files": existing_model_files,
        "missing_model_keys": missing_model_keys,
        "expected_model_keys": expected_model_keys if expected_model_keys is not None else [],
    }


def resolve_experiment_status(
    *,
    output_dir,
    experiment_name: str,
    model_name: str,
    targets: list[str],
    window_size: int,
    require_metrics: bool = True,
    require_probabilities: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Función genérica para cualquier modelo:
    - genera expected_model_keys
    - verifica si existen artefactos
    - define si hay que entrenar o se puede omitir

    Retorna un dict con:
    - should_skip
    - expected_model_keys
    - artifact_check
    - experiment_name
    """

    expected_model_keys = make_expected_model_keys(
        model_name=model_name,
        targets=targets,
        window_size=window_size,
    )

    artifact_check = check_experiment_artifacts_exist(
        output_dir=output_dir,
        experiment_name=experiment_name,
        expected_model_keys=expected_model_keys,
        require_metrics=require_metrics,
        require_probabilities=require_probabilities,
    )

    should_skip = artifact_check["exists_all"]

    if verbose:
        print("\n" + "=" * 80)
        print("EXPERIMENT CHECK")
        print("=" * 80)
        print(f"experiment_name       = {experiment_name}")
        print(f"model_name            = {model_name}")
        print(f"window_size           = L{int(window_size)}")
        print(f"targets               = {targets}")
        print(f"artifact_dir          = {artifact_check['artifact_dir']}")
        print(f"models_dir_exists     = {artifact_check['models_dir_exists']}")
        print(f"metrics_exists        = {artifact_check['metrics_exists']}")
        print(f"probabilities_exists  = {artifact_check['probabilities_exists']}")
        print(f"metadata_exists       = {artifact_check['metadata_exists']}")
        print(f"n_expected_models     = {len(expected_model_keys)}")
        print(f"n_missing_models      = {len(artifact_check['missing_model_keys'])}")
        print(f"should_skip           = {should_skip}")

        if artifact_check["missing_model_keys"]:
            print("\n[MISSING MODEL KEYS]")
            for key in artifact_check["missing_model_keys"]:
                print(f" - {key}")

    return {
        "should_skip": should_skip,
        "expected_model_keys": expected_model_keys,
        "artifact_check": artifact_check,
        "experiment_name": experiment_name,
    }

## **7.4. Carga de artefactos**

In [20]:
from pathlib import Path
import json
import joblib
import pandas as pd


def load_model_artifacts(
    *,
    output_dir,
    experiment_name: str,
    load_metrics: bool = True,
    load_probabilities: bool = True,
    load_metadata: bool = True,
    verbose: bool = True,
) -> dict:
    """
    Carga artefactos previamente guardados por save_model_artifacts().

    Retorna
    -------
    dict con:
    - "models"
    - "metrics"
    - "probabilities"
    - "metadata"
    - "artifact_dir"
    """

    output_dir = Path(output_dir)
    artifact_dir = output_dir / experiment_name
    models_dir = artifact_dir / "models"

    if not artifact_dir.exists():
        raise FileNotFoundError(f"No existe el directorio del experimento: {artifact_dir}")

    if not models_dir.exists():
        raise FileNotFoundError(f"No existe el directorio de modelos: {models_dir}")

    # ==================================================
    # 1) Cargar modelos
    # ==================================================
    models = {}
    model_files = sorted(models_dir.glob("*.joblib"))

    if not model_files:
        raise FileNotFoundError(f"No se encontraron modelos .joblib en: {models_dir}")

    for model_path in model_files:
        model_key = model_path.stem
        models[model_key] = joblib.load(model_path)

    # ==================================================
    # 2) Cargar métricas
    # ==================================================
    metrics = pd.DataFrame()
    metrics_path = artifact_dir / "metrics.parquet"
    if load_metrics:
        if not metrics_path.exists():
            raise FileNotFoundError(f"No existe metrics.parquet: {metrics_path}")
        metrics = pd.read_parquet(metrics_path)

    # ==================================================
    # 3) Cargar probabilidades
    # ==================================================
    probabilities = pd.DataFrame()
    probabilities_path = artifact_dir / "probabilities.parquet"
    if load_probabilities:
        if not probabilities_path.exists():
            raise FileNotFoundError(f"No existe probabilities.parquet: {probabilities_path}")
        probabilities = pd.read_parquet(probabilities_path)

    # ==================================================
    # 4) Cargar metadata
    # ==================================================
    metadata = {}
    metadata_path = artifact_dir / "metadata.json"
    if load_metadata:
        if not metadata_path.exists():
            raise FileNotFoundError(f"No existe metadata.json: {metadata_path}")
        with open(metadata_path, "r", encoding="utf-8") as f:
            metadata = json.load(f)

    if verbose:
        print("\n" + "=" * 80)
        print("ARTEFACTOS CARGADOS")
        print("=" * 80)
        print(f"artifact_dir = {artifact_dir}")
        print(f"n_models     = {len(models)}")
        print(f"metrics_rows = {len(metrics)}")
        print(f"prob_rows    = {len(probabilities)}")

    return {
        "artifact_dir": str(artifact_dir),
        "models": models,
        "metrics": metrics,
        "probabilities": probabilities,
        "metadata": metadata,
    }

## **8. Gestión de dispositivo y memoria**

In [21]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [22]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-25 00:30:43,600 | INFO | Seeds fijadas en 42


# **10. Modelo LR (Logistic Regression)**

## **Hiperparámetros seleccionados**

In [23]:
# =========================================================
# Logistic Regression | Configuración final de hiperparámetros
# =========================================================

lr_config = {
    "model_name": "logistic_regression",

    "model_params": {
        "C": 0.01,
        "max_iter": 1000,
        "solver": "lbfgs",
        "multi_class": "multinomial",
        "class_weight": "balanced",
        "random_state": SEED,
        "input_mode": "2d_flat",
    },

    "decision_rules": {
        "balanced": {
            "threshold_long": 0.40,
            "threshold_short": 0.45,
        },
        "conservative": {
            "threshold_long": 0.45,
            "threshold_short": 0.45,
        },
        "baseline": {
            "threshold_long": 0.40,
            "threshold_short": 0.40,
        },
    },
}

## **10.1. Función unitaria por bundle**

In [24]:
from sklearn.linear_model import LogisticRegression

def run_logistic_for_bundle_seq2one(
    bundle,
    *,
    multi_class="multinomial",
    solver="lbfgs",
    max_iter=1000,
    C=1.0,
    random_state=42,
    input_mode="2d_flat",
    class_weight=None,
    n_jobs=None,
    verbose=False,
):
    """
    Entrena Logistic Regression para un bundle seq2one
    usando TRAIN y genera predicciones sobre VALID.

    Esta función:
    - prepara X_train / X_valid
    - entrena el modelo
    - devuelve el modelo entrenado
    - devuelve predicciones de clase y probabilidades en VALID

    No aplica thresholds de decisión; eso se resuelve después
    en la etapa de evaluación operativa.
    """

    # =========================
    # 1. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    target = bundle.get("target")
    horizon = bundle.get("horizon")
    window_size = bundle.get("window_size")

    # =========================
    # 2. PREPARAR INPUT
    # =========================
    X_train_model = prepare_X_for_model(X_train, input_mode=input_mode)
    X_valid_model = prepare_X_for_model(X_valid, input_mode=input_mode)

    # =========================
    # 3. MODELO
    # =========================
    model = LogisticRegression(
        multi_class=multi_class,
        solver=solver,
        max_iter=max_iter,
        C=C,
        random_state=random_state,
        class_weight=class_weight,
        n_jobs=n_jobs,
    )

    # =========================
    # 4. TRAIN
    # =========================
    model.fit(X_train_model, y_train)

    # =========================
    # 5. PREDICT (VALID)
    # =========================
    y_pred_valid = model.predict(X_valid_model)
    y_proba_valid = model.predict_proba(X_valid_model)

    if verbose:
        print(
            f"[LOGISTIC] target={target} | horizon={horizon} | "
            f"window_size={window_size} | "
            f"X_train={X_train_model.shape} | X_valid={X_valid_model.shape} | "
            f"solver={solver} | C={C} | class_weight={class_weight}"
        )

    return {
        "model_name": "logistic_regression",
        "target": target,
        "horizon": horizon,
        "window_size": window_size,
        "input_mode": input_mode,
        "multi_class": multi_class,
        "solver": solver,
        "max_iter": max_iter,
        "C": C,
        "random_state": random_state,
        "class_weight": class_weight,
        "n_jobs": n_jobs,
        "X_train_shape": X_train_model.shape,
        "X_valid_shape": X_valid_model.shape,
        "classes_": model.classes_.tolist(),
        "model": model,
        "y_valid": y_valid,
        "y_pred_valid": y_pred_valid,
        "y_proba_valid": y_proba_valid,
    }

## **10.2. Función de evaluación sobre uno o más bundles**

In [25]:
from typing import Any, Dict, List, Sequence, Union
import pandas as pd


def eval_logistic_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    model_name: str = "logistic_regression",
    max_iter: int = 1000,
    random_state: int = 42,
    C: float = 1.0,
    multi_class: str = "multinomial",
    solver: str = "lbfgs",
    input_mode: str = "2d_flat",
    class_weight=None,
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
    verbose: bool = False,
):
    """
    Evalúa Logistic Regression para uno o varios bundles seq2one
    usando VALID.

    Retorna
    -------
    dict con:
    - "models": dict de modelos entrenados
    - "metrics": DataFrame de métricas
    - "probabilities": DataFrame de outputs probabilísticos / decisión
    """

    # ==================================================
    # 1) Normalizar entrada
    # ==================================================
    if isinstance(bundles, dict):
        if "train" in bundles and "valid" in bundles:
            bundles_list: List[Dict[str, Any]] = [bundles]
        else:
            bundles_list = list(bundles.values())
    else:
        bundles_list = list(bundles)

    metrics_rows = []
    probabilities_rows = []
    models_dict = {}

    # ==================================================
    # 2) Iterar por bundles
    # ==================================================
    for bundle in bundles_list:
        target = bundle.get("target")
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"model={model_name} | "
                f"class_weight={class_weight}"
            )

        # ----------------------------------------------
        # 3) Entrenar + predecir
        # ----------------------------------------------
        preds = run_logistic_for_bundle_seq2one(
            bundle,
            multi_class=multi_class,
            solver=solver,
            max_iter=max_iter,
            C=C,
            random_state=random_state,
            input_mode=input_mode,
            class_weight=class_weight,
            verbose=False,
        )

        model_key = f"{model_name}__{target}__L{window_size}"
        models_dict[model_key] = preds["model"]

        y_true = preds["y_valid"]
        y_pred = preds["y_pred_valid"]
        y_proba = preds["y_proba_valid"]
        model_classes = preds["classes_"]

        expected_classes = [-1, 0, 1]
        if list(model_classes) != expected_classes:
            raise ValueError(
                f"Orden de clases inesperado para predict_proba. "
                f"Esperado={expected_classes}, obtenido={model_classes}"
            )

        experiment_meta = {
            "model": model_name,
            "split": "valid",
            "window_size": window_size,
            "target": target,
            "horizon": horizon,
            "class_weight_mode": "balanced" if class_weight == "balanced" else "none",
            "input_mode": input_mode,
            "C": C,
            "max_iter": max_iter,
            "multi_class": multi_class,
            "solver": solver,
            "random_state": random_state,
            "threshold_long": prob_threshold_long,
            "threshold_short": prob_threshold_short,
        }

        # ----------------------------------------------
        # 4) Métricas
        # ----------------------------------------------
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split="valid",
            target=target,
            labels=expected_classes,
        )

        df_metrics_row = classification_metrics_to_df(
            metrics,
            model=model_name,
            split="valid",
            window_size=window_size,
            target=target,
            horizon=horizon,
        )

        for k, v in experiment_meta.items():
            df_metrics_row[k] = v

        metrics_rows.append(df_metrics_row)

        # ----------------------------------------------
        # 5) Probabilidades + decisión
        # ----------------------------------------------
        proba_df = compute_probabilistic_outputs(
            y_proba=y_proba,
            class_labels=expected_classes,
            y_true=y_true,
        )

        decision_df = apply_decision_rule(
            proba_df,
            long_class=1,
            short_class=-1,
            long_threshold=prob_threshold_long,
            short_threshold=prob_threshold_short,
        )

        overlap_cols = [c for c in decision_df.columns if c in proba_df.columns]
        if overlap_cols:
            decision_df = decision_df.drop(columns=overlap_cols)

        df_prob = pd.concat(
            [proba_df.reset_index(drop=True), decision_df.reset_index(drop=True)],
            axis=1,
        )

        for k, v in experiment_meta.items():
            df_prob[k] = v

        df_prob["class_labels"] = str(expected_classes)

        probabilities_rows.append(df_prob)

    # ==================================================
    # 6) Consolidar salida
    # ==================================================
    df_metrics_all = (
        pd.concat(metrics_rows, ignore_index=True)
        if metrics_rows else pd.DataFrame()
    )

    df_probabilities_all = (
        pd.concat(probabilities_rows, ignore_index=True)
        if probabilities_rows else pd.DataFrame()
    )

    return {
        "models": models_dict,
        "metrics": df_metrics_all,
        "probabilities": df_probabilities_all,
    }

## **10.3. Función orquestadora por `window_size`**

In [26]:
import gc
import pandas as pd


def run_logistic(
    window_size: int,
    *,
    targets: list[str] | None = None,
    experiment_name: str,
    output_dir,
    save_artifacts: bool = True,
    force_retrain: bool = False,
    verbose: bool = True,
    model_name: str = "logistic_regression",
    max_iter: int = 1000,
    random_state: int = 42,
    C: float = 1.0,
    multi_class: str = "multinomial",
    solver: str = "lbfgs",
    input_mode: str = "2d_flat",
    class_weight=None,
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
):
    """
    Ejecuta Logistic Regression para una sola window_size
    sobre los targets T2 indicados.

    Flujo:
    - si existen artefactos y force_retrain=False -> carga y devuelve
    - si no existen -> entrena, evalúa, guarda y devuelve

    Retorna
    -------
    dict con:
    - "models"
    - "metrics"
    - "probabilities"
    - "metadata"
    - "artifact_dir" (si carga o guarda)
    """

    if targets is None:
        targets = TARGETS

    if not targets:
        raise ValueError("La lista de targets no puede estar vacía.")

    size = int(window_size)

    expected_model_keys = [
        f"{model_name}__{target}__L{size}"
        for target in targets
    ]

    # ==================================================
    # 0) Check artefactos existentes
    # ==================================================
    if not force_retrain:
        check = check_experiment_artifacts_exist(
            output_dir=output_dir,
            experiment_name=experiment_name,
            expected_model_keys=expected_model_keys,
            require_metrics=True,
            require_probabilities=True,
        )

        if check["exists_all"]:
            if verbose:
                print("\n" + "=" * 80)
                print("EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS")
                print("=" * 80)
                print(f"experiment_name = {experiment_name}")
                print(f"artifact_dir    = {check['artifact_dir']}")

            loaded = load_model_artifacts(
                output_dir=output_dir,
                experiment_name=experiment_name,
                load_metrics=True,
                load_probabilities=True,
                load_metadata=True,
                verbose=verbose,
            )

            return loaded

    bundles = None
    results = None

    try:
        # ==================================================
        # 1) Encabezado
        # ==================================================
        if verbose:
            print("\n" + "=" * 80)
            print(f"LOGISTIC REGRESSION | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"experiment_name = {experiment_name}")
            print(f"targets         = {targets}")
            print(f"class_weight    = {class_weight}")
            print(f"input_mode      = {input_mode}")
            print(f"C               = {C}")
            print(f"max_iter        = {max_iter}")
            print(f"solver          = {solver}")
            print(f"multi_class     = {multi_class}")
            print(f"random_state    = {random_state}")
            print(f"thr_long        = {prob_threshold_long}")
            print(f"thr_short       = {prob_threshold_short}")

        # ==================================================
        # 2) Construcción de bundles
        # ==================================================
        if verbose:
            print(f"\n[BUILD] L{size} | n_targets={len(targets)}")

        bundles = create_bundles(
            window_size=size,
            targets=targets,
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        # ==================================================
        # 3) Entrenamiento + evaluación VALID
        # ==================================================
        if verbose:
            print(
                f"\n[EVAL] L{size} | split=valid | "
                f"model={model_name} | class_weight={class_weight}"
            )

        results = eval_logistic_bundles(
            bundles=bundles,
            model_name=model_name,
            max_iter=max_iter,
            random_state=random_state,
            C=C,
            multi_class=multi_class,
            solver=solver,
            input_mode=input_mode,
            class_weight=class_weight,
            prob_threshold_long=prob_threshold_long,
            prob_threshold_short=prob_threshold_short,
            verbose=verbose,
        )

        models_dict = results["models"]

        df_metrics = results["metrics"]
        if not df_metrics.empty:
            sort_cols_metrics = [
                c for c in ["window_size", "target", "split", "horizon", "model"]
                if c in df_metrics.columns
            ]
            if sort_cols_metrics:
                df_metrics = (
                    df_metrics
                    .sort_values(sort_cols_metrics)
                    .reset_index(drop=True)
                )

        df_probabilities = results["probabilities"]
        if not df_probabilities.empty:
            sort_cols_prob = [
                c for c in ["window_size", "target", "split", "horizon", "model"]
                if c in df_probabilities.columns
            ]
            if sort_cols_prob:
                df_probabilities = (
                    df_probabilities
                    .sort_values(sort_cols_prob)
                    .reset_index(drop=True)
                )

        metadata = {
            "model_family": model_name,
            "targets": targets,
            "window_size": size,
            "C": C,
            "max_iter": max_iter,
            "solver": solver,
            "multi_class": multi_class,
            "input_mode": input_mode,
            "class_weight": class_weight,
            "random_state": random_state,
            "threshold_long": prob_threshold_long,
            "threshold_short": prob_threshold_short,
        }

        save_paths = None
        if save_artifacts:
            save_paths = save_model_artifacts(
                models=models_dict,
                metrics=df_metrics,
                probabilities=df_probabilities,
                output_dir=output_dir,
                experiment_name=experiment_name,
                metadata=metadata,
                overwrite=True,
                verbose=verbose,
            )

        # ==================================================
        # 4) Resumen final
        # ==================================================
        if verbose:
            print(
                f"\n[DONE] L{size} | "
                f"models={len(models_dict)} | "
                f"metrics_rows={len(df_metrics)} | "
                f"probabilities_rows={len(df_probabilities)}"
            )

        return {
            "models": models_dict,
            "metrics": df_metrics,
            "probabilities": df_probabilities,
            "metadata": metadata,
            "artifact_dir": save_paths["artifact_dir"] if save_paths is not None else None,
            "save_paths": save_paths,
        }

    finally:
        del bundles, results
        gc.collect()

## **10.4. Ejecución de entrenamientos**

### **Carga de hiperparámetros**

In [27]:
lr_model_params = lr_config["model_params"]
lr_rule_balanced = lr_config["decision_rules"]["balanced"]
lr_rule_conservative = lr_config["decision_rules"]["conservative"]
lr_rule_baseline = lr_config["decision_rules"]["baseline"]

### **LR baseline**

In [28]:
results_lr_baseline = run_logistic(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    experiment_name="lr_final_baseline",
    output_dir=OUTPUT_DIR_BEST_MODEL,
    C=lr_model_params["C"],
    max_iter=lr_model_params["max_iter"],
    solver=lr_model_params["solver"],
    multi_class=lr_model_params["multi_class"],
    class_weight=lr_model_params["class_weight"],
    random_state=lr_model_params["random_state"],
    input_mode=lr_model_params["input_mode"],
    prob_threshold_long=lr_rule_baseline["threshold_long"],
    prob_threshold_short=lr_rule_baseline["threshold_short"],
    verbose=True,
)


EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS
experiment_name = lr_final_baseline
artifact_dir    = /content/drive/MyDrive/neural_profit/best_models/lr_final_baseline

ARTEFACTOS CARGADOS
artifact_dir = /content/drive/MyDrive/neural_profit/best_models/lr_final_baseline
n_models     = 2
metrics_rows = 2
prob_rows    = 13764


### **LR balanced**

In [29]:
results_lr_balanced = run_logistic(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    experiment_name="lr_final_balanced",
    output_dir=OUTPUT_DIR_BEST_MODEL,
    C=lr_model_params["C"],
    max_iter=lr_model_params["max_iter"],
    solver=lr_model_params["solver"],
    multi_class=lr_model_params["multi_class"],
    class_weight=lr_model_params["class_weight"],
    random_state=lr_model_params["random_state"],
    input_mode=lr_model_params["input_mode"],
    prob_threshold_long=lr_rule_balanced["threshold_long"],
    prob_threshold_short=lr_rule_balanced["threshold_short"],
    verbose=True,
)


EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS
experiment_name = lr_final_balanced
artifact_dir    = /content/drive/MyDrive/neural_profit/best_models/lr_final_balanced

ARTEFACTOS CARGADOS
artifact_dir = /content/drive/MyDrive/neural_profit/best_models/lr_final_balanced
n_models     = 2
metrics_rows = 2
prob_rows    = 13764


### **LR conservative**

In [30]:
results_lr_conservative = run_logistic(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    experiment_name="lr_final_conservative",
    output_dir=OUTPUT_DIR_BEST_MODEL,
    C=lr_model_params["C"],
    max_iter=lr_model_params["max_iter"],
    solver=lr_model_params["solver"],
    multi_class=lr_model_params["multi_class"],
    class_weight=lr_model_params["class_weight"],
    random_state=lr_model_params["random_state"],
    input_mode=lr_model_params["input_mode"],
    prob_threshold_long=lr_rule_conservative["threshold_long"],
    prob_threshold_short=lr_rule_conservative["threshold_short"],
    verbose=True,
)


EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS
experiment_name = lr_final_conservative
artifact_dir    = /content/drive/MyDrive/neural_profit/best_models/lr_final_conservative

ARTEFACTOS CARGADOS
artifact_dir = /content/drive/MyDrive/neural_profit/best_models/lr_final_conservative
n_models     = 2
metrics_rows = 2
prob_rows    = 13764


# **11. Modelo GRU**

## **Hiperparámetros seleccionados**

In [31]:
# =========================================================
# GRU | Configuración final de hiperparámetros
# =========================================================

gru_config = {
    "model_name": "gru",

    # -----------------------------------------------------
    # Configuraciones de modelo (arquitectura + entrenamiento)
    # -----------------------------------------------------
    "model_params": {

        # Configuración base (benchmark predictivo)
        "baseline": {
            "hidden_size": 128,
            "num_layers": 2,
            "learning_rate": 5e-4,
            "dropout": 0.0,
            "batch_size": 2048,
            "grad_clip_norm": 0.5,
        },

        # Configuración operativa (alta precisión)
        "conservative": {
            "hidden_size": 256,
            "num_layers": 1,
            "learning_rate": 1e-3,
            "dropout": 0.25,  # valor intermedio dentro de 0.2–0.3
            "batch_size": 4096,
            "grad_clip_norm": 0.5,
        },
    },

    # -----------------------------------------------------
    # Reglas de decisión (thresholds)
    # -----------------------------------------------------
    "decision_rules": {

        # Benchmark (alineado con evaluación clásica)
        "baseline": {
            "threshold_long": 0.40,
            "threshold_short": 0.40,
        },

        # Configuración operativa (alta precisión)
        "conservative": {
            "threshold_long": 0.45,
            "threshold_short": 0.45,
        },
    },
}

## **11.1. Función unitaria por bundle**

In [32]:
import copy
import numpy as np
import torch
import torch.nn as nn


class GRUClassifier(nn.Module):
    def __init__(
        self,
        n_features: int,
        hidden_size: int = 128,
        num_layers: int = 1,
        dropout: float = 0.0,
        num_classes: int = 3,
    ):
        super().__init__()

        gru_dropout = dropout if num_layers > 1 else 0.0

        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=gru_dropout,
        )

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.gru(x)
        last_out = out[:, -1, :]
        last_out = self.dropout(last_out)
        logits = self.fc(last_out)
        return logits


def run_gru_for_bundle_seq2one(
    bundle,
    *,
    hidden_size: int = 128,
    num_layers: int = 2,
    learning_rate: float = 5e-4,
    dropout: float = 0.0,
    batch_size: int = 2048,
    grad_clip_norm: float = 0.5,
    weight_decay: float = 0.0,
    epochs: int = 20,
    patience: int = 5,
    class_weight: str | None = "balanced",
    random_state: int = 42,
    device: str | None = None,
    deterministic: bool = True,
    num_workers: int = 0,
    verbose: bool = False,
):
    """
    Entrena un modelo GRU seq2one para un bundle.

    Esta función:
    - entrena sobre TRAIN
    - evalúa sobre VALID
    - devuelve modelo entrenado
    - devuelve predicciones y probabilidades en VALID
    - no aplica thresholds operativos
    """

    # =========================
    # 1. SEEDS Y DEVICE
    # =========================
    torch.manual_seed(random_state)
    np.random.seed(random_state)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    use_pin_memory = device == "cuda"

    # =========================
    # 2. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    target = bundle.get("target")
    horizon = bundle.get("horizon")
    window_size = bundle.get("window_size")

    n_features = X_train.shape[2]

    # =========================
    # 3. LABEL ENCODING
    # =========================
    classes_ = np.sort(np.unique(y_train))
    expected_classes = np.array([-1, 0, 1])

    if not np.array_equal(classes_, expected_classes):
        raise ValueError(
            f"Clases inesperadas en y_train. "
            f"Esperado={expected_classes.tolist()}, obtenido={classes_.tolist()}"
        )

    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int64)
    y_valid_enc = np.array([class_to_idx[y] for y in y_valid], dtype=np.int64)

    num_classes = len(classes_)

    # =========================
    # 4. CLASS WEIGHTS
    # =========================
    criterion_weight = None

    if class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_classes)
        total = counts.sum()
        weights = total / (num_classes * counts)

        criterion_weight = torch.tensor(
            weights,
            dtype=torch.float32,
            device=device,
        )

    # =========================
    # 5. TENSORES
    # =========================
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_enc, dtype=torch.long)

    X_valid_t = torch.tensor(X_valid, dtype=torch.float32)
    y_valid_t = torch.tensor(y_valid_enc, dtype=torch.long)

    # =========================
    # 6. DATALOADERS
    # =========================
    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train_t, y_train_t),
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    valid_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_valid_t, y_valid_t),
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=use_pin_memory,
    )

    # =========================
    # 7. MODELO
    # =========================
    model = GRUClassifier(
        n_features=n_features,
        hidden_size=hidden_size,
        num_layers=num_layers,
        dropout=dropout,
        num_classes=num_classes,
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=criterion_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    # =========================
    # HELPERS
    # =========================
    def _move(x):
        return x.to(device, non_blocking=True) if device == "cuda" else x.to(device)

    def compute_valid_loss():
        model.eval()
        total_loss = 0.0
        n = 0

        with torch.no_grad():
            for xb, yb in valid_loader:
                xb, yb = _move(xb), _move(yb)
                logits = model(xb)
                loss = criterion(logits, yb)

                total_loss += loss.item() * xb.size(0)
                n += xb.size(0)

        return total_loss / n

    def predict_proba(loader):
        model.eval()
        logits_all = []

        with torch.no_grad():
            for xb, _ in loader:
                xb = _move(xb)
                logits = model(xb)
                logits_all.append(logits.cpu())

        logits_all = torch.cat(logits_all, dim=0)
        proba = torch.softmax(logits_all, dim=1).numpy()

        return proba

    # =========================
    # 8. TRAIN
    # =========================
    best_state = None
    best_loss = np.inf
    wait = 0

    for epoch in range(epochs):
        model.train()

        for xb, yb in train_loader:
            xb, yb = _move(xb), _move(yb)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()

            if grad_clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    grad_clip_norm,
                )

            optimizer.step()

        val_loss = compute_valid_loss()

        if verbose:
            print(
                f"[GRU] target={target} | epoch={epoch + 1:02d} | "
                f"valid_loss={val_loss:.6f}"
            )

        if val_loss < best_loss:
            best_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    # =========================
    # 9. PREDICT VALID
    # =========================
    y_proba_valid = predict_proba(valid_loader)

    y_pred_valid_idx = y_proba_valid.argmax(axis=1)
    y_pred_valid = np.array(
        [idx_to_class[i] for i in y_pred_valid_idx],
        dtype=int,
    )

    if verbose:
        print(
            f"[GRU DONE] target={target} | horizon={horizon} | "
            f"window_size={window_size} | best_valid_loss={best_loss:.6f}"
        )

    return {
        "model_name": "gru",
        "target": target,
        "horizon": horizon,
        "window_size": window_size,
        "hidden_size": hidden_size,
        "num_layers": num_layers,
        "learning_rate": learning_rate,
        "dropout": dropout,
        "batch_size": batch_size,
        "grad_clip_norm": grad_clip_norm,
        "weight_decay": weight_decay,
        "epochs": epochs,
        "patience": patience,
        "class_weight": class_weight,
        "random_state": random_state,
        "device": device,
        "n_features": n_features,
        "classes_": classes_.tolist(),
        "best_valid_loss": best_loss,
        "model": model,
        "y_valid": y_valid,
        "y_pred_valid": y_pred_valid,
        "y_proba_valid": y_proba_valid,
    }

## **11.2. Función de evaluación sobre bundle**

In [33]:
import gc
import pandas as pd
import torch


def eval_gru_bundles(
    bundles,
    *,
    model_name: str = "gru",
    hidden_size: int = 128,
    num_layers: int = 2,
    learning_rate: float = 5e-4,
    dropout: float = 0.0,
    batch_size: int = 2048,
    grad_clip_norm: float = 0.5,
    weight_decay: float = 0.0,
    epochs: int = 20,
    patience: int = 5,
    class_weight: str | None = "balanced",
    random_state: int = 42,
    device: str | None = None,
    num_workers: int = 0,
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
    verbose: bool = False,
):
    """
    Entrena y evalúa GRU para uno o varios bundles seq2one.

    Retorna:
    - models
    - metrics
    - probabilities
    """

    # =========================
    # 1) Normalizar entrada
    # =========================
    if isinstance(bundles, dict):
        if "train" in bundles and "valid" in bundles:
            bundles_list = [bundles]
        else:
            bundles_list = list(bundles.values())
    else:
        bundles_list = list(bundles)

    models_dict = {}
    metrics_rows = []
    probabilities_rows = []

    expected_classes = [-1, 0, 1]

    # =========================
    # 2) Loop por bundle
    # =========================
    for bundle in bundles_list:
        target = bundle.get("target")
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> GRU | L{window_size} | target={target} | "
                f"hidden_size={hidden_size} | num_layers={num_layers} | "
                f"dropout={dropout} | batch_size={batch_size}"
            )

        preds = None

        try:
            # =========================
            # 3) Entrenar modelo
            # =========================
            preds = run_gru_for_bundle_seq2one(
                bundle,
                hidden_size=hidden_size,
                num_layers=num_layers,
                learning_rate=learning_rate,
                dropout=dropout,
                batch_size=batch_size,
                grad_clip_norm=grad_clip_norm,
                weight_decay=weight_decay,
                epochs=epochs,
                patience=patience,
                class_weight=class_weight,
                random_state=random_state,
                device=device,
                num_workers=num_workers,
                verbose=verbose,
            )

            model_key = f"{model_name}__{target}__L{window_size}"
            models_dict[model_key] = preds["model"]

            y_true = preds["y_valid"]
            y_pred = preds["y_pred_valid"]
            y_proba = preds["y_proba_valid"]
            model_classes = preds["classes_"]

            if list(model_classes) != expected_classes:
                raise ValueError(
                    f"Orden de clases inesperado para predict_proba. "
                    f"Esperado={expected_classes}, obtenido={model_classes}"
                )

            experiment_meta = {
                "model": model_name,
                "split": "valid",
                "window_size": window_size,
                "target": target,
                "horizon": horizon,
                "hidden_size": hidden_size,
                "num_layers": num_layers,
                "learning_rate": learning_rate,
                "dropout": dropout,
                "batch_size": batch_size,
                "grad_clip_norm": grad_clip_norm,
                "weight_decay": weight_decay,
                "epochs": epochs,
                "patience": patience,
                "class_weight_mode": "balanced" if class_weight == "balanced" else "none",
                "random_state": random_state,
                "threshold_long": prob_threshold_long,
                "threshold_short": prob_threshold_short,
                "best_valid_loss": preds["best_valid_loss"],
            }

            # =========================
            # 4) Métricas clasificación
            # =========================
            metrics = compute_classification_metrics(
                y_true=y_true,
                y_pred=y_pred,
                model_name=model_name,
                split="valid",
                target=target,
                labels=expected_classes,
            )

            df_metrics_row = classification_metrics_to_df(
                metrics,
                model=model_name,
                split="valid",
                window_size=window_size,
                target=target,
                horizon=horizon,
            )

            for k, v in experiment_meta.items():
                df_metrics_row[k] = v

            metrics_rows.append(df_metrics_row)

            # =========================
            # 5) Probabilidades + decisión
            # =========================
            proba_df = compute_probabilistic_outputs(
                y_proba=y_proba,
                class_labels=expected_classes,
                y_true=y_true,
            )

            decision_df = apply_decision_rule(
                proba_df,
                long_class=1,
                short_class=-1,
                long_threshold=prob_threshold_long,
                short_threshold=prob_threshold_short,
            )

            overlap_cols = [c for c in decision_df.columns if c in proba_df.columns]
            if overlap_cols:
                decision_df = decision_df.drop(columns=overlap_cols)

            df_prob = pd.concat(
                [proba_df.reset_index(drop=True), decision_df.reset_index(drop=True)],
                axis=1,
            )

            for k, v in experiment_meta.items():
                df_prob[k] = v

            df_prob["class_labels"] = str(expected_classes)

            probabilities_rows.append(df_prob)

        finally:
            if preds is not None:
                del preds

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # =========================
    # 6) Consolidar salida
    # =========================
    df_metrics_all = (
        pd.concat(metrics_rows, ignore_index=True)
        if metrics_rows else pd.DataFrame()
    )

    df_probabilities_all = (
        pd.concat(probabilities_rows, ignore_index=True)
        if probabilities_rows else pd.DataFrame()
    )

    return {
        "models": models_dict,
        "metrics": df_metrics_all,
        "probabilities": df_probabilities_all,
    }

## **11.3. Función orquestadora**

In [34]:
import gc
import torch


def run_gru(
    window_size: int,
    *,
    targets: list[str] | None = None,
    experiment_name: str,
    output_dir,
    save_artifacts: bool = True,
    force_retrain: bool = False,
    verbose: bool = True,
    model_name: str = "gru",
    hidden_size: int = 128,
    num_layers: int = 2,
    learning_rate: float = 5e-4,
    dropout: float = 0.0,
    batch_size: int = 2048,
    grad_clip_norm: float = 0.5,
    weight_decay: float = 0.0,
    epochs: int = 20,
    patience: int = 5,
    class_weight: str | None = "balanced",
    random_state: int = 42,
    device: str | None = None,
    num_workers: int = 0,
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
):
    """
    Orquestador final para GRU seq2one.

    Flujo:
    - si existen artefactos y force_retrain=False -> carga y devuelve
    - si no existen -> entrena, evalúa, guarda y devuelve
    """

    if targets is None:
        targets = ["t2_p40_h30", "t2_p50_h30"]

    if not targets:
        raise ValueError("La lista de targets no puede estar vacía.")

    size = int(window_size)

    expected_model_keys = [
        f"{model_name}__{target}__L{size}"
        for target in targets
    ]

    # ==================================================
    # 0) Check artefactos existentes
    # ==================================================
    if not force_retrain:
        check = check_experiment_artifacts_exist(
            output_dir=output_dir,
            experiment_name=experiment_name,
            expected_model_keys=expected_model_keys,
            require_metrics=True,
            require_probabilities=True,
        )

        if check["exists_all"]:
            if verbose:
                print("\n" + "=" * 80)
                print("EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS")
                print("=" * 80)
                print(f"experiment_name = {experiment_name}")
                print(f"artifact_dir    = {check['artifact_dir']}")

            return load_model_artifacts(
                output_dir=output_dir,
                experiment_name=experiment_name,
                load_metrics=True,
                load_probabilities=True,
                load_metadata=True,
                verbose=verbose,
            )

    bundles = None
    results = None

    try:
        # ==================================================
        # 1) Encabezado
        # ==================================================
        if verbose:
            print("\n" + "=" * 80)
            print(f"GRU | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"experiment_name = {experiment_name}")
            print(f"targets         = {targets}")
            print(f"hidden_size     = {hidden_size}")
            print(f"num_layers      = {num_layers}")
            print(f"learning_rate   = {learning_rate}")
            print(f"dropout         = {dropout}")
            print(f"batch_size      = {batch_size}")
            print(f"grad_clip_norm  = {grad_clip_norm}")
            print(f"class_weight    = {class_weight}")
            print(f"thr_long        = {prob_threshold_long}")
            print(f"thr_short       = {prob_threshold_short}")

        # ==================================================
        # 2) Construcción de bundles
        # ==================================================
        bundles = create_bundles(
            window_size=size,
            targets=targets,
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        # ==================================================
        # 3) Entrenamiento + evaluación VALID
        # ==================================================
        results = eval_gru_bundles(
            bundles=bundles,
            model_name=model_name,
            hidden_size=hidden_size,
            num_layers=num_layers,
            learning_rate=learning_rate,
            dropout=dropout,
            batch_size=batch_size,
            grad_clip_norm=grad_clip_norm,
            weight_decay=weight_decay,
            epochs=epochs,
            patience=patience,
            class_weight=class_weight,
            random_state=random_state,
            device=device,
            num_workers=num_workers,
            prob_threshold_long=prob_threshold_long,
            prob_threshold_short=prob_threshold_short,
            verbose=verbose,
        )

        models_dict = results["models"]

        df_metrics = results["metrics"]
        if not df_metrics.empty:
            sort_cols = [
                c for c in ["window_size", "target", "split", "horizon", "model"]
                if c in df_metrics.columns
            ]
            if sort_cols:
                df_metrics = df_metrics.sort_values(sort_cols).reset_index(drop=True)

        df_probabilities = results["probabilities"]
        if not df_probabilities.empty:
            sort_cols = [
                c for c in ["window_size", "target", "split", "horizon", "model"]
                if c in df_probabilities.columns
            ]
            if sort_cols:
                df_probabilities = df_probabilities.sort_values(sort_cols).reset_index(drop=True)

        metadata = {
            "model_family": model_name,
            "targets": targets,
            "window_size": size,
            "hidden_size": hidden_size,
            "num_layers": num_layers,
            "learning_rate": learning_rate,
            "dropout": dropout,
            "batch_size": batch_size,
            "grad_clip_norm": grad_clip_norm,
            "weight_decay": weight_decay,
            "epochs": epochs,
            "patience": patience,
            "class_weight": class_weight,
            "random_state": random_state,
            "threshold_long": prob_threshold_long,
            "threshold_short": prob_threshold_short,
        }

        save_paths = None
        if save_artifacts:
            save_paths = save_model_artifacts(
                models=models_dict,
                metrics=df_metrics,
                probabilities=df_probabilities,
                output_dir=output_dir,
                experiment_name=experiment_name,
                metadata=metadata,
                overwrite=True,
                verbose=verbose,
            )

        if verbose:
            print(
                f"\n[DONE] L{size} | "
                f"models={len(models_dict)} | "
                f"metrics_rows={len(df_metrics)} | "
                f"probabilities_rows={len(df_probabilities)}"
            )

        return {
            "models": models_dict,
            "metrics": df_metrics,
            "probabilities": df_probabilities,
            "metadata": metadata,
            "artifact_dir": save_paths["artifact_dir"] if save_paths is not None else None,
            "save_paths": save_paths,
        }

    finally:
        del bundles, results
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## **11.4. Ejecución de entrenamientos**

### **Configuración base (benchmark predictivo)**

In [35]:
gru_params_baseline = gru_config["model_params"]["baseline"]
gru_rule_baseline = gru_config["decision_rules"]["baseline"]

results_gru_baseline = run_gru(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    experiment_name="gru_final_baseline",
    output_dir=OUTPUT_DIR_BEST_MODEL,
    **gru_params_baseline,
    prob_threshold_long=gru_rule_baseline["threshold_long"],
    prob_threshold_short=gru_rule_baseline["threshold_short"],
    random_state=SEED,
    verbose=True,
)


EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS
experiment_name = gru_final_baseline
artifact_dir    = /content/drive/MyDrive/neural_profit/best_models/gru_final_baseline

ARTEFACTOS CARGADOS
artifact_dir = /content/drive/MyDrive/neural_profit/best_models/gru_final_baseline
n_models     = 2
metrics_rows = 2
prob_rows    = 13764


### **Configuración operativa (alta precisión)**

In [36]:
gru_params_conservative = gru_config["model_params"]["conservative"]
gru_rule_conservative = gru_config["decision_rules"]["conservative"]

results_gru_conservative = run_gru(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    experiment_name="gru_final_conservative",
    output_dir=OUTPUT_DIR_BEST_MODEL,
    **gru_params_conservative,
    prob_threshold_long=gru_rule_conservative["threshold_long"],
    prob_threshold_short=gru_rule_conservative["threshold_short"],
    random_state=SEED,
    verbose=True,
)


EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS
experiment_name = gru_final_conservative
artifact_dir    = /content/drive/MyDrive/neural_profit/best_models/gru_final_conservative

ARTEFACTOS CARGADOS
artifact_dir = /content/drive/MyDrive/neural_profit/best_models/gru_final_conservative
n_models     = 2
metrics_rows = 2
prob_rows    = 13764


# **12. Modelo XGBoost**

## **Hiperparámetros seleccionados**

In [37]:
# =========================================================
# XGBoost | Configuración final de hiperparámetros
# =========================================================

xgb_config = {
    "model_name": "xgboost",

    # -----------------------------------------------------
    # Configuración de modelo
    # -----------------------------------------------------
    "model_params": {

        # Configuración final predictiva
        "baseline": {
            "n_estimators": 600,
            "max_depth": 3,
            "learning_rate": 0.01,
            "subsample": 1.0,
            "colsample_bytree": 0.8,
            "gamma": 0.0,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
        },

        # Misma configuración predictiva, distinta regla operativa
        "conservative": {
            "n_estimators": 600,
            "max_depth": 3,
            "learning_rate": 0.01,
            "subsample": 1.0,
            "colsample_bytree": 0.8,
            "gamma": 0.0,
            "reg_alpha": 0.1,
            "reg_lambda": 1.0,
        },
    },

    # -----------------------------------------------------
    # Reglas de decisión (thresholds)
    # -----------------------------------------------------
    "decision_rules": {

        # Configuración balanceada
        "baseline": {
            "threshold_long": 0.45,
            "threshold_short": 0.40,
        },

        # Configuración conservadora
        "conservative": {
            "threshold_long": 0.50,
            "threshold_short": 0.50,
        },
    },
}

## **12.1. Función unitaria por bundle**

In [38]:
from xgboost import XGBClassifier
import numpy as np


def run_xgboost_for_bundle_seq2one(
    bundle,
    *,
    n_estimators: int = 600,
    max_depth: int = 3,
    learning_rate: float = 0.01,
    subsample: float = 1.0,
    colsample_bytree: float = 0.8,
    min_child_weight: int = 1,
    gamma: float = 0.0,
    reg_alpha: float = 0.1,
    reg_lambda: float = 1.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
    tree_method: str = "hist",
    device: str | None = None,
    verbose: bool = False,
):
    """
    Entrena XGBoost para un bundle seq2one.

    Esta función:
    - entrena sobre TRAIN
    - predice sobre VALID
    - devuelve modelo entrenado
    - devuelve clases, predicciones y probabilidades
    """

    # =========================
    # 1. DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    y_valid = bundle["valid"]["y"]

    target = bundle.get("target")
    horizon = bundle.get("horizon")
    window_size = bundle.get("window_size")

    # =========================
    # 2. INPUT
    # =========================
    X_train_model = prepare_X_for_model(X_train, input_mode=input_mode)
    X_valid_model = prepare_X_for_model(X_valid, input_mode=input_mode)

    # =========================
    # 3. LABEL ENCODING
    # =========================
    classes_ = np.sort(np.unique(y_train))
    expected_classes = np.array([-1, 0, 1])

    if not np.array_equal(classes_, expected_classes):
        raise ValueError(
            f"Clases inesperadas. "
            f"Esperado={expected_classes.tolist()}, obtenido={classes_.tolist()}"
        )

    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int32)
    num_class = len(classes_)

    # =========================
    # 4. SAMPLE WEIGHTS
    # =========================
    sample_weight = None
    weights_by_idx = None

    if class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_class)
        total = counts.sum()

        if np.any(counts == 0):
            raise ValueError(f"Hay clases sin muestras en TRAIN. counts={counts.tolist()}")

        weights = total / (num_class * counts)
        weights_by_idx = {idx: float(w) for idx, w in enumerate(weights)}
        sample_weight = weights[y_train_enc].astype(np.float32)

    elif isinstance(class_weight, dict):
        weights_by_idx = {
            class_to_idx[k]: float(v)
            for k, v in class_weight.items()
            if k in class_to_idx
        }
        sample_weight = np.array(
            [weights_by_idx.get(i, 1.0) for i in y_train_enc],
            dtype=np.float32,
        )

    elif class_weight is not None:
        raise ValueError("class_weight debe ser None, 'balanced' o dict")

    # =========================
    # 5. DEVICE
    # =========================
    if device is None:
        device = "cuda"

    # =========================
    # 6. MODEL
    # =========================
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=num_class,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        gamma=gamma,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=random_state,
        n_jobs=n_jobs,
        eval_metric="mlogloss",
        tree_method=tree_method,
        device=device,
        verbosity=1 if verbose else 0,
    )

    # =========================
    # 7. TRAIN
    # =========================
    model.fit(
        X_train_model,
        y_train_enc,
        sample_weight=sample_weight,
    )

    # =========================
    # 8. PREDICT VALID
    # =========================
    y_pred_valid_enc = model.predict(X_valid_model)
    y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])

    y_proba_valid = model.predict_proba(X_valid_model)

    if verbose:
        print(
            f"[XGBOOST] target={target} | horizon={horizon} | "
            f"window_size={window_size} | "
            f"n_estimators={n_estimators} | max_depth={max_depth} | "
            f"learning_rate={learning_rate}"
        )

    # =========================
    # 9. RETURN
    # =========================
    return {
        "model_name": "xgboost",
        "target": target,
        "horizon": horizon,
        "window_size": window_size,
        "input_mode": input_mode,
        "class_weight": class_weight,
        "tree_method": tree_method,
        "device": device,

        # hiperparámetros
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "min_child_weight": min_child_weight,
        "gamma": gamma,
        "reg_alpha": reg_alpha,
        "reg_lambda": reg_lambda,
        "random_state": random_state,
        "n_jobs": n_jobs,

        # encoding
        "classes_": classes_.tolist(),
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "weights_by_idx": weights_by_idx,

        # modelo y outputs
        "model": model,
        "y_valid": y_valid,
        "y_pred_valid": y_pred_valid,
        "y_proba_valid": y_proba_valid,
    }

## **12.2. Función de evaluación sobre bundle**

In [39]:
from typing import Any, Dict, List, Sequence, Union
import pandas as pd


def eval_xgboost_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    model_name: str = "xgboost",
    n_estimators: int = 600,
    max_depth: int = 3,
    learning_rate: float = 0.01,
    subsample: float = 1.0,
    colsample_bytree: float = 0.8,
    min_child_weight: float = 1.0,
    gamma: float = 0.0,
    reg_alpha: float = 0.1,
    reg_lambda: float = 1.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
    tree_method: str = "hist",
    device: str | None = None,
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
    verbose: bool = False,
):
    """
    Entrena y evalúa XGBoost para uno o varios bundles seq2one.

    Retorna:
    - models
    - metrics
    - probabilities
    """

    # ==================================================
    # 1) Normalizar entrada
    # ==================================================
    if isinstance(bundles, dict):
        if "train" in bundles and "valid" in bundles:
            bundles_list: List[Dict[str, Any]] = [bundles]
        else:
            bundles_list = list(bundles.values())
    else:
        bundles_list = list(bundles)

    metrics_rows = []
    probabilities_rows = []
    models_dict = {}

    expected_classes = [-1, 0, 1]

    # ==================================================
    # 2) Iterar por bundles
    # ==================================================
    for bundle in bundles_list:
        target = bundle.get("target")
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> XGB | L{window_size} | target={target} | "
                f"n_estimators={n_estimators} | max_depth={max_depth} | "
                f"learning_rate={learning_rate}"
            )

        preds = run_xgboost_for_bundle_seq2one(
            bundle,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            min_child_weight=min_child_weight,
            gamma=gamma,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
            tree_method=tree_method,
            device=device,
            verbose=False,
        )

        model_key = f"{model_name}__{target}__L{window_size}"
        models_dict[model_key] = preds["model"]

        y_true = preds["y_valid"]
        y_pred = preds["y_pred_valid"]
        y_proba = preds["y_proba_valid"]
        class_labels = preds["classes_"]

        if list(class_labels) != expected_classes:
            raise ValueError(
                f"Orden de clases inesperado para predict_proba. "
                f"Esperado={expected_classes}, obtenido={class_labels}"
            )

        if class_weight == "balanced":
            class_weight_mode = "balanced"
        elif class_weight is None:
            class_weight_mode = "none"
        else:
            class_weight_mode = "custom"

        experiment_meta = {
            "model": model_name,
            "split": "valid",
            "window_size": window_size,
            "target": target,
            "horizon": horizon,
            "class_weight_mode": class_weight_mode,
            "input_mode": input_mode,
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "min_child_weight": min_child_weight,
            "gamma": gamma,
            "reg_alpha": reg_alpha,
            "reg_lambda": reg_lambda,
            "tree_method": tree_method,
            "device": device,
            "n_jobs": n_jobs,
            "random_state": random_state,
            "threshold_long": prob_threshold_long,
            "threshold_short": prob_threshold_short,
        }

        # ==================================================
        # 3) Métricas de clasificación
        # ==================================================
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split="valid",
            target=target,
            labels=expected_classes,
        )

        df_metrics_row = classification_metrics_to_df(
            metrics,
            model=model_name,
            split="valid",
            window_size=window_size,
            target=target,
            horizon=horizon,
        )

        for k, v in experiment_meta.items():
            df_metrics_row[k] = v

        metrics_rows.append(df_metrics_row)

        # ==================================================
        # 4) Probabilidades + regla de decisión
        # ==================================================
        proba_df = compute_probabilistic_outputs(
            y_proba=y_proba,
            class_labels=expected_classes,
            y_true=y_true,
        )

        decision_df = apply_decision_rule(
            proba_df,
            long_class=1,
            short_class=-1,
            long_threshold=prob_threshold_long,
            short_threshold=prob_threshold_short,
        )

        overlap_cols = [c for c in decision_df.columns if c in proba_df.columns]
        if overlap_cols:
            decision_df = decision_df.drop(columns=overlap_cols)

        df_prob = pd.concat(
            [proba_df.reset_index(drop=True), decision_df.reset_index(drop=True)],
            axis=1,
        )

        for k, v in experiment_meta.items():
            df_prob[k] = v

        df_prob["class_labels"] = str(expected_classes)

        probabilities_rows.append(df_prob)

    # ==================================================
    # 5) Consolidar salida
    # ==================================================
    df_metrics_all = (
        pd.concat(metrics_rows, ignore_index=True)
        if metrics_rows else pd.DataFrame()
    )

    df_probabilities_all = (
        pd.concat(probabilities_rows, ignore_index=True)
        if probabilities_rows else pd.DataFrame()
    )

    return {
        "models": models_dict,
        "metrics": df_metrics_all,
        "probabilities": df_probabilities_all,
    }

## **12.3. Función orquestadora**

In [40]:
import gc


def run_xgboost(
    window_size: int,
    *,
    targets: list[str] | None = None,
    experiment_name: str,
    output_dir,
    save_artifacts: bool = True,
    force_retrain: bool = False,
    verbose: bool = True,
    model_name: str = "xgboost",
    n_estimators: int = 600,
    max_depth: int = 3,
    learning_rate: float = 0.01,
    subsample: float = 1.0,
    colsample_bytree: float = 0.8,
    min_child_weight: float = 1.0,
    gamma: float = 0.0,
    reg_alpha: float = 0.1,
    reg_lambda: float = 1.0,
    class_weight=None,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    tree_method: str = "hist",
    device: str | None = None,
    prob_threshold_long: float = 0.40,
    prob_threshold_short: float = 0.40,
):
    """
    Orquestador final para XGBoost seq2one.

    Flujo:
    - si existen artefactos y force_retrain=False -> carga y devuelve
    - si no existen -> entrena, evalúa, guarda y devuelve
    """

    if targets is None:
        targets = ["t2_p40_h30", "t2_p50_h30"]

    if not targets:
        raise ValueError("La lista de targets no puede estar vacía.")

    size = int(window_size)

    expected_model_keys = [
        f"{model_name}__{target}__L{size}"
        for target in targets
    ]

    # ==================================================
    # 0) Check artefactos existentes
    # ==================================================
    if not force_retrain:
        check = check_experiment_artifacts_exist(
            output_dir=output_dir,
            experiment_name=experiment_name,
            expected_model_keys=expected_model_keys,
            require_metrics=True,
            require_probabilities=True,
        )

        if check["exists_all"]:
            if verbose:
                print("\n" + "=" * 80)
                print("EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS")
                print("=" * 80)
                print(f"experiment_name = {experiment_name}")
                print(f"artifact_dir    = {check['artifact_dir']}")

            return load_model_artifacts(
                output_dir=output_dir,
                experiment_name=experiment_name,
                load_metrics=True,
                load_probabilities=True,
                load_metadata=True,
                verbose=verbose,
            )

    bundles = None
    results = None

    try:
        # ==================================================
        # 1) Encabezado
        # ==================================================
        if verbose:
            print("\n" + "=" * 80)
            print(f"XGBOOST | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"experiment_name  = {experiment_name}")
            print(f"targets          = {targets}")
            print(f"n_estimators     = {n_estimators}")
            print(f"max_depth        = {max_depth}")
            print(f"learning_rate    = {learning_rate}")
            print(f"subsample        = {subsample}")
            print(f"colsample_bytree = {colsample_bytree}")
            print(f"min_child_weight = {min_child_weight}")
            print(f"gamma            = {gamma}")
            print(f"reg_alpha        = {reg_alpha}")
            print(f"reg_lambda       = {reg_lambda}")
            print(f"class_weight     = {class_weight}")
            print(f"thr_long         = {prob_threshold_long}")
            print(f"thr_short        = {prob_threshold_short}")

        # ==================================================
        # 2) Construcción de bundles
        # ==================================================
        bundles = create_bundles(
            window_size=size,
            targets=targets,
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        # ==================================================
        # 3) Entrenamiento + evaluación VALID
        # ==================================================
        results = eval_xgboost_bundles(
            bundles=bundles,
            model_name=model_name,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            min_child_weight=min_child_weight,
            gamma=gamma,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
            tree_method=tree_method,
            device=device,
            prob_threshold_long=prob_threshold_long,
            prob_threshold_short=prob_threshold_short,
            verbose=verbose,
        )

        models_dict = results["models"]

        df_metrics = results["metrics"]
        if not df_metrics.empty:
            sort_cols = [
                c for c in ["window_size", "target", "split", "horizon", "model"]
                if c in df_metrics.columns
            ]
            if sort_cols:
                df_metrics = df_metrics.sort_values(sort_cols).reset_index(drop=True)

        df_probabilities = results["probabilities"]
        if not df_probabilities.empty:
            sort_cols = [
                c for c in ["window_size", "target", "split", "horizon", "model"]
                if c in df_probabilities.columns
            ]
            if sort_cols:
                df_probabilities = df_probabilities.sort_values(sort_cols).reset_index(drop=True)

        metadata = {
            "model_family": model_name,
            "targets": targets,
            "window_size": size,
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "min_child_weight": min_child_weight,
            "gamma": gamma,
            "reg_alpha": reg_alpha,
            "reg_lambda": reg_lambda,
            "class_weight": class_weight,
            "random_state": random_state,
            "n_jobs": n_jobs,
            "input_mode": input_mode,
            "tree_method": tree_method,
            "device": device,
            "threshold_long": prob_threshold_long,
            "threshold_short": prob_threshold_short,
        }

        save_paths = None
        if save_artifacts:
            save_paths = save_model_artifacts(
                models=models_dict,
                metrics=df_metrics,
                probabilities=df_probabilities,
                output_dir=output_dir,
                experiment_name=experiment_name,
                metadata=metadata,
                overwrite=True,
                verbose=verbose,
            )

        if verbose:
            print(
                f"\n[DONE] L{size} | "
                f"models={len(models_dict)} | "
                f"metrics_rows={len(df_metrics)} | "
                f"probabilities_rows={len(df_probabilities)}"
            )

        return {
            "models": models_dict,
            "metrics": df_metrics,
            "probabilities": df_probabilities,
            "metadata": metadata,
            "artifact_dir": save_paths["artifact_dir"] if save_paths is not None else None,
            "save_paths": save_paths,
        }

    finally:
        del bundles, results
        gc.collect()

## **12.4. Ejecución de entrenamientos**

### **Configuración base**

In [41]:
xgb_params_baseline = xgb_config["model_params"]["baseline"]
xgb_rule_baseline = xgb_config["decision_rules"]["baseline"]

results_xgb_baseline = run_xgboost(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    experiment_name="xgb_final_baseline",
    output_dir=OUTPUT_DIR_BEST_MODEL,
    **xgb_params_baseline,
    prob_threshold_long=xgb_rule_baseline["threshold_long"],
    prob_threshold_short=xgb_rule_baseline["threshold_short"],
    random_state=SEED,
    verbose=True,
)


EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS
experiment_name = xgb_final_baseline
artifact_dir    = /content/drive/MyDrive/neural_profit/best_models/xgb_final_baseline

ARTEFACTOS CARGADOS
artifact_dir = /content/drive/MyDrive/neural_profit/best_models/xgb_final_baseline
n_models     = 2
metrics_rows = 2
prob_rows    = 13764


### **Configuración conservative**

In [42]:
xgb_params_conservative = xgb_config["model_params"]["conservative"]
xgb_rule_conservative = xgb_config["decision_rules"]["conservative"]

results_xgb_conservative = run_xgboost(
    window_size=30,
    targets=["t2_p40_h30", "t2_p50_h30"],
    experiment_name="xgb_final_conservative",
    output_dir=OUTPUT_DIR_BEST_MODEL,
    **xgb_params_conservative,
    prob_threshold_long=xgb_rule_conservative["threshold_long"],
    prob_threshold_short=xgb_rule_conservative["threshold_short"],
    random_state=SEED,
    verbose=True,
)


EXPERIMENTO YA EXISTE -> CARGANDO ARTEFACTOS
experiment_name = xgb_final_conservative
artifact_dir    = /content/drive/MyDrive/neural_profit/best_models/xgb_final_conservative

ARTEFACTOS CARGADOS
artifact_dir = /content/drive/MyDrive/neural_profit/best_models/xgb_final_conservative
n_models     = 2
metrics_rows = 2
prob_rows    = 13764


# **13. Consolidar métricas**

In [43]:
# =========================================================
# 13. Consolidación de resultados finales en memoria
# =========================================================

import pandas as pd

# ---------------------------------------------------------
# 1) Diccionario de experimentos ejecutados
# ---------------------------------------------------------

final_results = {
    "lr_final_baseline": results_lr_baseline,
    "lr_final_balanced": results_lr_balanced,
    "lr_final_conservative": results_lr_conservative,
    "gru_final_baseline": results_gru_baseline,
    "gru_final_conservative": results_gru_conservative,
    "xgb_final_baseline": results_xgb_baseline,
    "xgb_final_conservative": results_xgb_conservative,
}

# ---------------------------------------------------------
# 2) Consolidar métricas
# ---------------------------------------------------------

metrics_frames = []

for experiment_name, result in final_results.items():
    df = result["metrics"].copy()
    df["experiment_name"] = experiment_name
    metrics_frames.append(df)

df_metrics_final = pd.concat(metrics_frames, ignore_index=True)

# ---------------------------------------------------------
# 3) Consolidar probabilidades
# ---------------------------------------------------------

probability_frames = []

for experiment_name, result in final_results.items():
    df = result["probabilities"].copy()
    df["experiment_name"] = experiment_name
    probability_frames.append(df)

df_probabilities_final = pd.concat(probability_frames, ignore_index=True)

# ---------------------------------------------------------
# 4) Ordenar tablas
# ---------------------------------------------------------

sort_cols_metrics = [
    c for c in [
        "experiment_name",
        "model",
        "target",
        "window_size",
        "horizon",
        "threshold_long",
        "threshold_short",
    ]
    if c in df_metrics_final.columns
]

df_metrics_final = (
    df_metrics_final
    .sort_values(sort_cols_metrics)
    .reset_index(drop=True)
)

sort_cols_prob = [
    c for c in [
        "experiment_name",
        "model",
        "target",
        "window_size",
        "horizon",
        "threshold_long",
        "threshold_short",
    ]
    if c in df_probabilities_final.columns
]

df_probabilities_final = (
    df_probabilities_final
    .sort_values(sort_cols_prob)
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# 5) Guardar consolidados
# ---------------------------------------------------------

metrics_final_path = OUTPUT_DIR_BEST_MODEL / "df_metrics_final_models.parquet"
probabilities_final_path = OUTPUT_DIR_BEST_MODEL / "df_probabilities_final_models.parquet"

df_metrics_final.to_parquet(metrics_final_path, index=False)
df_probabilities_final.to_parquet(probabilities_final_path, index=False)

# ---------------------------------------------------------
# 6) Resumen visual
# ---------------------------------------------------------

print("=" * 80)
print("CONSOLIDACIÓN FINAL COMPLETADA")
print("=" * 80)
print(f"Experimentos consolidados : {len(final_results)}")
print(f"Filas métricas            : {len(df_metrics_final)}")
print(f"Filas probabilidades      : {len(df_probabilities_final)}")
print(f"Métricas guardadas en     : {metrics_final_path}")
print(f"Probabilidades guardadas  : {probabilities_final_path}")

display_cols_metrics = [
    c for c in [
        "experiment_name",
        "model",
        "target",
        "window_size",
        "horizon",
        "threshold_long",
        "threshold_short",
        "balanced_accuracy",
        "f1_macro",
        "accuracy",
    ]
    if c in df_metrics_final.columns
]

display(df_metrics_final[display_cols_metrics])

CONSOLIDACIÓN FINAL COMPLETADA
Experimentos consolidados : 7
Filas métricas            : 14
Filas probabilidades      : 96348
Métricas guardadas en     : /content/drive/MyDrive/neural_profit/best_models/df_metrics_final_models.parquet
Probabilidades guardadas  : /content/drive/MyDrive/neural_profit/best_models/df_probabilities_final_models.parquet


,experiment_name,model,target,window_size,horizon,threshold_long,threshold_short,balanced_accuracy,f1_macro,accuracy
0,gru_final_baseline,gru,t2_p40_h30,30,30,0.40,0.40,0.420773,0.380090,0.393200
1,gru_final_baseline,gru,t2_p50_h30,30,30,0.40,0.40,0.413918,0.396479,0.418047
2,gru_final_conservative,gru,t2_p40_h30,30,30,0.45,0.45,0.404084,0.362182,0.374746
3,gru_final_conservative,gru,t2_p50_h30,30,30,0.45,0.45,0.403007,0.382632,0.411799
4,lr_final_balanced,logistic_regression,t2_p40_h30,30,30,0.40,0.45,0.406574,0.373965,0.382302
5,lr_final_balanced,logistic_regression,t2_p50_h30,30,30,0.40,0.45,0.405650,0.389273,0.415286
6,lr_final_baseline,logistic_regression,t2_p40_h30,30,30,0.40,0.40,0.406574,0.373965,0.382302
7,lr_final_baseline,logistic_regression,t2_p50_h30,30,30,0.40,0.40,0.405650,0.389273,0.415286
8,lr_final_conservative,logistic_regression,t2_p40_h30,30,30,0.45,0.45,0.406574,0.373965,0.382302
9,lr_final_conservative,logistic_regression,t2_p50_h30,30,30,0.45,0.45,0.405650,0.389273,0.415286


In [70]:
df_probabilities_final

,proba_-1,proba_0,proba_1,pred_label,confidence,y_true,is_correct,signal_raw,trade,model,...,max_depth,subsample,colsample_bytree,min_child_weight,gamma,reg_alpha,reg_lambda,tree_method,device,n_jobs
0,0.224806,0.504271,0.270923,0,0.504271,1,False,0,False,gru,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0.230340,0.502102,0.267557,0,0.502102,1,False,0,False,gru,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,0.236065,0.498919,0.265016,0,0.498919,1,False,0,False,gru,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,0.240946,0.497206,0.261848,0,0.497206,1,False,0,False,gru,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0.241786,0.500210,0.258004,0,0.500210,1,False,0,False,gru,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96343,0.281398,0.420800,0.297802,0,0.420800,0,True,0,False,xgboost,...,3.0,1.0,0.8,1.0,0.0,0.1,1.0,hist,None,-1.0
96344,0.247446,0.431741,0.320813,0,0.431741,0,True,0,False,xgboost,...,3.0,1.0,0.8,1.0,0.0,0.1,1.0,hist,None,-1.0
96345,0.246289,0.421395,0.332316,0,0.421395,0,True,0,False,xgboost,...,3.0,1.0,0.8,1.0,0.0,0.1,1.0,hist,None,-1.0
96346,0.251025,0.425648,0.323327,0,0.425648,0,True,0,False,xgboost,...,3.0,1.0,0.8,1.0,0.0,0.1,1.0,hist,None,-1.0


In [69]:
df_metrics_final

,model,split,window_size,target,horizon,n_samples,accuracy,balanced_accuracy,f1_macro,f1_weighted,...,max_depth,subsample,colsample_bytree,min_child_weight,gamma,reg_alpha,reg_lambda,tree_method,device,n_jobs
0,gru,valid,30,t2_p40_h30,30,6882,0.393200,0.420773,0.380090,0.367872,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,gru,valid,30,t2_p50_h30,30,6882,0.418047,0.413918,0.396479,0.398675,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,gru,valid,30,t2_p40_h30,30,6882,0.374746,0.404084,0.362182,0.350577,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,gru,valid,30,t2_p50_h30,30,6882,0.411799,0.403007,0.382632,0.386842,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,logistic_regression,valid,30,t2_p40_h30,30,6882,0.382302,0.406574,0.373965,0.367032,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,logistic_regression,valid,30,t2_p50_h30,30,6882,0.415286,0.405650,0.389273,0.394089,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,logistic_regression,valid,30,t2_p40_h30,30,6882,0.382302,0.406574,0.373965,0.367032,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,logistic_regression,valid,30,t2_p50_h30,30,6882,0.415286,0.405650,0.389273,0.394089,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,logistic_regression,valid,30,t2_p40_h30,30,6882,0.382302,0.406574,0.373965,0.367032,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,logistic_regression,valid,30,t2_p50_h30,30,6882,0.415286,0.405650,0.389273,0.394089,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# **14. Análisis consolidado**

En esta etapa se comparan de forma conjunta los modelos finales entrenados con los hiperparámetros seleccionados durante el hypertuning. El objetivo ya no es ajustar configuraciones, sino evaluar de manera homogénea el desempeño final de Logistic Regression, GRU y XGBoost sobre los mismos targets seleccionados.

El análisis se divide en dos dimensiones principales. Primero, se comparan las métricas predictivas tradicionales, como balanced_accuracy, f1_macro y accuracy, para identificar qué modelo aprende mejor la estructura del problema. Luego, se analizan las probabilidades y reglas operativas, evaluando la frecuencia de señales, la precisión útil y la relación entre calidad y volumen de operaciones.

Esta comparación permite distinguir entre el mejor modelo desde el punto de vista estadístico y el mejor modelo desde el punto de vista operativo, lo cual es fundamental para un sistema de señales aplicado a trading.

## **Código de análisis conjunto**

In [44]:
# =========================================================
# 14. Comparación consolidada de modelos finales
# =========================================================

import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1) Validaciones iniciales
# ---------------------------------------------------------

if "df_metrics_final" not in globals():
    raise NameError("No existe df_metrics_final. Ejecutá primero la consolidación de métricas.")

if "df_probabilities_final" not in globals():
    raise NameError("No existe df_probabilities_final. Ejecutá primero la consolidación de probabilidades.")

dfm = df_metrics_final.copy()
dfp = df_probabilities_final.copy()

print("=" * 100)
print("COMPARACIÓN CONSOLIDADA DE MODELOS FINALES")
print("=" * 100)
print(f"Filas métricas      : {len(dfm)}")
print(f"Filas probabilidades: {len(dfp)}")

# ---------------------------------------------------------
# 2) Columnas base disponibles
# ---------------------------------------------------------

base_cols = [
    c for c in [
        "experiment_name",
        "model",
        "target",
        "window_size",
        "horizon",
        "threshold_long",
        "threshold_short",
    ]
    if c in dfm.columns
]

group_cols = [
    c for c in [
        "experiment_name",
        "model",
        "target",
        "window_size",
        "horizon",
        "threshold_long",
        "threshold_short",
    ]
    if c in dfp.columns
]

if not group_cols:
    raise ValueError("No se encontraron columnas suficientes para agrupar df_probabilities_final.")

# ---------------------------------------------------------
# 3) Ranking predictivo
# ---------------------------------------------------------

predictive_cols = [
    c for c in [
        "balanced_accuracy",
        "f1_macro",
        "accuracy",
        "f1_weighted",
        "balanced_accuracy_gain_vs_naive",
        "f1_macro_gain_vs_naive",
        "f1_weighted_gain_vs_naive",
    ]
    if c in dfm.columns
]

df_predictive_ranking = (
    dfm[base_cols + predictive_cols]
    .sort_values(
        [c for c in ["balanced_accuracy", "f1_macro", "accuracy"] if c in dfm.columns],
        ascending=False,
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("1) RANKING PREDICTIVO")
print("=" * 100)
display(df_predictive_ranking)

# ---------------------------------------------------------
# 4) Detectar columnas de y_true y decisión operativa
# ---------------------------------------------------------

possible_y_true_cols = ["y_true", "true_label", "target_true", "label", "y"]
possible_decision_cols = [
    "decision",
    "trade",
    "signal",
    "signal_raw",
    "pred_signal",
    "y_decision",
    "y_pred_decision",
    "prediction",
]

y_true_col = "y_true"
decision_col = "trade"

if y_true_col not in dfp.columns:
    raise ValueError(f"No existe columna {y_true_col}")

if decision_col not in dfp.columns:
    raise ValueError(f"No existe columna {decision_col}")

print("\nColumnas detectadas:")
print(f"y_true   -> {y_true_col}")
print(f"decision -> {decision_col}")

# ---------------------------------------------------------
# 5) Métricas operativas desde probabilidades
# ---------------------------------------------------------

dfp["_is_trade"] = dfp[decision_col].isin([-1, 1])
dfp["_is_long"] = dfp[decision_col].eq(1)
dfp["_is_short"] = dfp[decision_col].eq(-1)

dfp["_is_useful"] = (
    dfp["_is_trade"] &
    dfp[y_true_col].isin([-1, 1]) &
    dfp[decision_col].eq(dfp[y_true_col])
)

dfp["_is_wrong_trade"] = (
    dfp["_is_trade"] &
    dfp[y_true_col].isin([-1, 0, 1]) &
    ~dfp[decision_col].eq(dfp[y_true_col])
)

df_operational_summary = (
    dfp
    .groupby(group_cols, dropna=False)
    .agg(
        n_samples=(decision_col, "size"),
        n_trades=("_is_trade", "sum"),
        n_long=("_is_long", "sum"),
        n_short=("_is_short", "sum"),
        n_useful=("_is_useful", "sum"),
        n_wrong_trades=("_is_wrong_trade", "sum"),
    )
    .reset_index()
)

df_operational_summary["trade_rate"] = (
    df_operational_summary["n_trades"] / df_operational_summary["n_samples"]
)

df_operational_summary["useful_rate_total"] = (
    df_operational_summary["n_useful"] / df_operational_summary["n_samples"]
)

df_operational_summary["precision_useful"] = np.where(
    df_operational_summary["n_trades"] > 0,
    df_operational_summary["n_useful"] / df_operational_summary["n_trades"],
    np.nan,
)

df_operational_summary["long_share"] = np.where(
    df_operational_summary["n_trades"] > 0,
    df_operational_summary["n_long"] / df_operational_summary["n_trades"],
    np.nan,
)

df_operational_summary["short_share"] = np.where(
    df_operational_summary["n_trades"] > 0,
    df_operational_summary["n_short"] / df_operational_summary["n_trades"],
    np.nan,
)

df_operational_ranking = (
    df_operational_summary
    .sort_values(
        ["precision_useful", "useful_rate_total", "trade_rate"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("2) RANKING OPERATIVO")
print("=" * 100)

display_cols_operational = [
    c for c in [
        "experiment_name",
        "model",
        "target",
        "threshold_long",
        "threshold_short",
        "n_samples",
        "n_trades",
        "trade_rate",
        "n_useful",
        "useful_rate_total",
        "precision_useful",
        "n_long",
        "n_short",
        "long_share",
        "short_share",
    ]
    if c in df_operational_ranking.columns
]

display(df_operational_ranking[display_cols_operational])

# ---------------------------------------------------------
# 6) Comparación consolidada predictiva + operativa
# ---------------------------------------------------------

merge_keys = [
    c for c in [
        "experiment_name",
        "model",
        "target",
        "window_size",
        "horizon",
        "threshold_long",
        "threshold_short",
    ]
    if c in dfm.columns and c in df_operational_summary.columns
]

df_model_comparison = dfm.merge(
    df_operational_summary,
    on=merge_keys,
    how="left",
)

ranking_cols = [
    c for c in [
        "balanced_accuracy",
        "f1_macro",
        "precision_useful",
        "useful_rate_total",
        "trade_rate",
    ]
    if c in df_model_comparison.columns
]

df_model_comparison = (
    df_model_comparison
    .sort_values(ranking_cols, ascending=False)
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("3) COMPARACIÓN CONSOLIDADA PREDICTIVA + OPERATIVA")
print("=" * 100)

display_cols_comparison = [
    c for c in [
        "experiment_name",
        "model",
        "target",
        "threshold_long",
        "threshold_short",
        "balanced_accuracy",
        "f1_macro",
        "accuracy",
        "n_trades",
        "trade_rate",
        "n_useful",
        "useful_rate_total",
        "precision_useful",
    ]
    if c in df_model_comparison.columns
]

display(df_model_comparison[display_cols_comparison])

# ---------------------------------------------------------
# 7) Comparación por target
# ---------------------------------------------------------

print("\n" + "=" * 100)
print("4) COMPARACIÓN POR TARGET")
print("=" * 100)

for target in sorted(df_model_comparison["target"].dropna().unique()):
    print("\n" + "-" * 100)
    print(f"TARGET: {target}")
    print("-" * 100)

    df_target = (
        df_model_comparison[df_model_comparison["target"] == target]
        .sort_values(
            [c for c in ["balanced_accuracy", "precision_useful", "useful_rate_total"] if c in df_model_comparison.columns],
            ascending=False,
        )
        .reset_index(drop=True)
    )

    display(df_target[display_cols_comparison])

# ---------------------------------------------------------
# 8) Selección automática de candidatos
# ---------------------------------------------------------

# Mejor benchmark predictivo
predictive_rank_cols = [
    c for c in ["balanced_accuracy", "f1_macro", "accuracy"]
    if c in df_model_comparison.columns
]

df_best_predictive = (
    df_model_comparison
    .sort_values(predictive_rank_cols, ascending=False)
    .head(1)
    .reset_index(drop=True)
)

# Mejor operativo por precisión útil
df_best_precision = (
    df_model_comparison
    .sort_values(["precision_useful", "useful_rate_total", "trade_rate"], ascending=False)
    .head(1)
    .reset_index(drop=True)
)

# Mejor equilibrio señal / ruido
df_balance_candidates = df_model_comparison.copy()

for col in ["balanced_accuracy", "f1_macro", "precision_useful", "useful_rate_total", "trade_rate"]:
    if col in df_balance_candidates.columns:
        min_v = df_balance_candidates[col].min()
        max_v = df_balance_candidates[col].max()
        if max_v > min_v:
            df_balance_candidates[f"{col}_norm"] = (
                (df_balance_candidates[col] - min_v) / (max_v - min_v)
            )
        else:
            df_balance_candidates[f"{col}_norm"] = 0.0

score_components = [
    c for c in [
        "balanced_accuracy_norm",
        "f1_macro_norm",
        "precision_useful_norm",
        "useful_rate_total_norm",
    ]
    if c in df_balance_candidates.columns
]

df_balance_candidates["final_balance_score"] = df_balance_candidates[score_components].mean(axis=1)

df_best_balance = (
    df_balance_candidates
    .sort_values("final_balance_score", ascending=False)
    .head(1)
    .reset_index(drop=True)
)

print("\n" + "=" * 100)
print("5) CANDIDATOS FINALES")
print("=" * 100)

print("\n[MEJOR BENCHMARK PREDICTIVO]")
display(df_best_predictive[display_cols_comparison])

print("\n[MEJOR CONFIGURACIÓN OPERATIVA POR PRECISIÓN ÚTIL]")
display(df_best_precision[display_cols_comparison])

print("\n[MEJOR EQUILIBRIO GENERAL]")
display_cols_balance = display_cols_comparison + ["final_balance_score"]
display(df_best_balance[[c for c in display_cols_balance if c in df_best_balance.columns]])

# ---------------------------------------------------------
# 9) Guardar tablas comparativas
# ---------------------------------------------------------

comparison_path = OUTPUT_DIR_BEST_MODEL / "df_model_comparison_final.parquet"
operational_path = OUTPUT_DIR_BEST_MODEL / "df_operational_summary_final.parquet"

df_model_comparison.to_parquet(comparison_path, index=False)
df_operational_summary.to_parquet(operational_path, index=False)

print("\n" + "=" * 100)
print("TABLAS COMPARATIVAS GUARDADAS")
print("=" * 100)
print(f"Comparación final : {comparison_path}")
print(f"Resumen operativo : {operational_path}")

COMPARACIÓN CONSOLIDADA DE MODELOS FINALES
Filas métricas      : 14
Filas probabilidades: 96348

1) RANKING PREDICTIVO


,experiment_name,model,target,window_size,horizon,threshold_long,threshold_short,balanced_accuracy,f1_macro,accuracy,f1_weighted,balanced_accuracy_gain_vs_naive,f1_macro_gain_vs_naive,f1_weighted_gain_vs_naive
0,gru_final_baseline,gru,t2_p40_h30,30,30,0.40,0.40,0.420773,0.380090,0.393200,0.367872,0.087439,0.191788,0.145505
1,gru_final_baseline,gru,t2_p50_h30,30,30,0.40,0.40,0.413918,0.396479,0.418047,0.398675,0.080585,0.220193,0.208556
2,xgb_final_baseline,xgboost,t2_p40_h30,30,30,0.45,0.40,0.408318,0.369286,0.384481,0.362860,0.074985,0.180984,0.140494
3,xgb_final_conservative,xgboost,t2_p40_h30,30,30,0.50,0.50,0.408318,0.369286,0.384481,0.362860,0.074985,0.180984,0.140494
4,lr_final_balanced,logistic_regression,t2_p40_h30,30,30,0.40,0.45,0.406574,0.373965,0.382302,0.367032,0.073241,0.185664,0.144665
5,lr_final_baseline,logistic_regression,t2_p40_h30,30,30,0.40,0.40,0.406574,0.373965,0.382302,0.367032,0.073241,0.185664,0.144665
6,lr_final_conservative,logistic_regression,t2_p40_h30,30,30,0.45,0.45,0.406574,0.373965,0.382302,0.367032,0.073241,0.185664,0.144665
7,lr_final_balanced,logistic_regression,t2_p50_h30,30,30,0.40,0.45,0.405650,0.389273,0.415286,0.394089,0.072317,0.212986,0.203971
8,lr_final_baseline,logistic_regression,t2_p50_h30,30,30,0.40,0.40,0.405650,0.389273,0.415286,0.394089,0.072317,0.212986,0.203971
9,lr_final_conservative,logistic_regression,t2_p50_h30,30,30,0.45,0.45,0.405650,0.389273,0.415286,0.394089,0.072317,0.212986,0.203971



Columnas detectadas:
y_true   -> y_true
decision -> trade

2) RANKING OPERATIVO


,experiment_name,model,target,threshold_long,threshold_short,n_samples,n_trades,trade_rate,n_useful,useful_rate_total,precision_useful,n_long,n_short,long_share,short_share
0,xgb_final_conservative,xgboost,t2_p50_h30,0.50,0.50,6882,102,0.014821,51,0.007411,0.500000,102,0,1.0,0.0
1,xgb_final_conservative,xgboost,t2_p40_h30,0.50,0.50,6882,175,0.025429,81,0.011770,0.462857,175,0,1.0,0.0
2,gru_final_conservative,gru,t2_p40_h30,0.45,0.45,6882,238,0.034583,110,0.015984,0.462185,238,0,1.0,0.0
3,xgb_final_baseline,xgboost,t2_p40_h30,0.45,0.40,6882,578,0.083987,265,0.038506,0.458478,578,0,1.0,0.0
4,xgb_final_baseline,xgboost,t2_p50_h30,0.45,0.40,6882,283,0.041122,129,0.018745,0.455830,283,0,1.0,0.0
5,lr_final_balanced,logistic_regression,t2_p40_h30,0.40,0.45,6882,1037,0.150683,459,0.066696,0.442623,1037,0,1.0,0.0
6,gru_final_conservative,gru,t2_p50_h30,0.45,0.45,6882,283,0.041122,123,0.017873,0.434629,283,0,1.0,0.0
7,lr_final_conservative,logistic_regression,t2_p40_h30,0.45,0.45,6882,621,0.090235,263,0.038216,0.423510,621,0,1.0,0.0
8,lr_final_baseline,logistic_regression,t2_p40_h30,0.40,0.40,6882,1458,0.211857,613,0.089073,0.420439,1458,0,1.0,0.0
9,lr_final_balanced,logistic_regression,t2_p50_h30,0.40,0.45,6882,1016,0.147632,415,0.060302,0.408465,1016,0,1.0,0.0



3) COMPARACIÓN CONSOLIDADA PREDICTIVA + OPERATIVA


...


[MEJOR CONFIGURACIÓN OPERATIVA POR PRECISIÓN ÚTIL]


,experiment_name,model,target,threshold_long,threshold_short,balanced_accuracy,f1_macro,accuracy,n_trades,trade_rate,n_useful,useful_rate_total,precision_useful
0,xgb_final_conservative,xgboost,t2_p50_h30,0.5,0.5,0.365294,0.275999,0.388695,102,0.014821,51,0.007411,0.5



[MEJOR EQUILIBRIO GENERAL]


,experiment_name,model,target,threshold_long,threshold_short,balanced_accuracy,f1_macro,accuracy,n_trades,trade_rate,n_useful,useful_rate_total,precision_useful,final_balance_score
0,gru_final_baseline,gru,t2_p40_h30,0.4,0.4,0.420773,0.38009,0.3932,1691,0.245713,686,0.09968,0.405677,0.769123



TABLAS COMPARATIVAS GUARDADAS
Comparación final : /content/drive/MyDrive/neural_profit/best_models/df_model_comparison_final.parquet
Resumen operativo : /content/drive/MyDrive/neural_profit/best_models/df_operational_summary_final.parquet


## **Conclusiones del análisis conjunto de modelos finales**


El análisis comparativo de los modelos entrenados permite establecer con claridad el comportamiento relativo de Logistic Regression, GRU y XGBoost bajo un mismo marco experimental, tanto desde el punto de vista predictivo como operativo.

En primer lugar, se observa una dominancia consistente del modelo GRU en términos predictivos. Este modelo presenta los mejores resultados en `balanced_accuracy` y `f1_macro` para ambos targets evaluados, manteniendo además una estabilidad destacable entre configuraciones. Esto indica que el GRU logra capturar de forma más efectiva la dinámica temporal del problema, lo cual es coherente con su arquitectura secuencial. Si bien la diferencia con otros modelos no es extrema, sí es sistemática, lo que le otorga mayor confiabilidad como modelo base.

En segundo lugar, XGBoost muestra un comportamiento asimétrico dependiendo del target. Mientras que en `t2_p40_h30` se mantiene competitivo, en `t2_p50_h30` presenta una caída significativa en todas las métricas relevantes. Esto sugiere que el modelo depende fuertemente de la densidad de señal del target, funcionando adecuadamente en escenarios más frecuentes, pero perdiendo capacidad cuando los eventos son más escasos o exigentes. En consecuencia, su robustez estructural es inferior a la de GRU.

Por su parte, Logistic Regression presenta un comportamiento altamente estable, con métricas prácticamente invariantes entre configuraciones de thresholds. Esto refleja que el modelo es robusto, pero con una capacidad limitada para capturar relaciones complejas. En la práctica, se posiciona como un buen baseline, pero no como candidato principal.

Desde el punto de vista operativo, se identifica un trade-off claro entre precisión y frecuencia de señales. Las configuraciones más conservadoras, especialmente en XGBoost y GRU, alcanzan niveles elevados de `precision_useful`, pero a costa de un `trade_rate` extremadamente bajo, lo que implica una operativa casi inexistente. Por el contrario, configuraciones más activas, como GRU baseline, presentan un mayor volumen de señales con una reducción moderada en la precisión, logrando un mejor balance global.

Un aspecto crítico del análisis es la presencia de un sesgo estructural en todas las configuraciones evaluadas: el sistema no genera señales bajistas. Todas las decisiones corresponden a posiciones largas, lo que se refleja en `n_short = 0` y `long_share = 1.0`. Este comportamiento puede estar asociado al dataset, a la definición del target o a los thresholds utilizados, y constituye una limitación importante que deberá abordarse en etapas posteriores.

En cuanto a la selección de modelos, los resultados permiten identificar tres perfiles claros. Como mejor benchmark predictivo, GRU baseline se posiciona como la opción más sólida, especialmente en `t2_p40_h30`. Como modelo de máxima precisión operativa, XGBoost en configuración conservadora alcanza los valores más altos de `precision_useful`, aunque con una frecuencia de operación muy baja. Finalmente, como mejor equilibrio entre capacidad predictiva y desempeño operativo, GRU baseline sobre `t2_p40_h30` emerge como la opción más robusta, combinando buen nivel de señal, volumen de operaciones y tasa de aciertos.

Adicionalmente, se observa que el target `t2_p40_h30` presenta un comportamiento más favorable en términos generales. Ofrece mayor densidad de señal, mayor estabilidad entre modelos y mejores resultados operativos. En contraste, `t2_p50_h30` resulta más exigente y menos estable, afectando especialmente a modelos como XGBoost.

En términos globales, el análisis evidencia que el valor del sistema no reside únicamente en el modelo utilizado, sino en la interacción entre modelo, target y regla de decisión basada en probabilidades. La definición de thresholds juega un rol central en la conversión de predicciones en señales operativas.

Como conclusión final, el sistema queda caracterizado por un modelo principal basado en GRU en su configuración baseline, operando sobre el target `t2_p40_h30` con thresholds 0.40 / 0.40. Este enfoque puede complementarse con XGBoost en configuración conservadora como filtro de alta confianza, dependiendo del diseño final de la estrategia.

# **15. Evaluación final out-of-sample en TEST**

En esta etapa se realiza la evaluación final out-of-sample (OOS) utilizando el conjunto de TEST, que no ha sido utilizado en ninguna fase previa de entrenamiento, tuning o selección de hiperparámetros.

El objetivo es validar si los resultados observados en VALID se mantienen fuera de muestra, tanto en términos predictivos como operativos. Esta evaluación permite medir la capacidad real de generalización de los modelos seleccionados y detectar posibles efectos de sobreajuste.

Se evalúan únicamente las configuraciones finales definidas en la etapa anterior:

- GRU baseline (modelo principal)
- XGBoost conservative (modelo de alta precisión)
- opcionalmente Logistic Regression como benchmark

El análisis se realiza replicando exactamente el mismo pipeline aplicado en VALID:

- predicciones sobre TEST
- cálculo de métricas de clasificación
- generación de probabilidades
- aplicación de reglas de decisión (thresholds)
- evaluación operativa

Con esto se obtiene una validación final del sistema antes de avanzar hacia la definición de la estrategia de trading.

## **15.1. Carga de modelos entrenados y seleccionados**

In [88]:
print(results_lr_balanced["models"].keys())
print(results_gru_baseline["models"].keys())
print(results_xgb_conservative["models"].keys())

dict_keys(['logistic_regression__t2_p40_h30__L30', 'logistic_regression__t2_p50_h30__L30'])
dict_keys(['gru__t2_p40_h30__L30', 'gru__t2_p50_h30__L30'])
dict_keys(['xgboost__t2_p40_h30__L30', 'xgboost__t2_p50_h30__L30'])


In [97]:
def get_model_from_results(results, model_name, target, window_size=30):
    model_key = f"{model_name}__{target}__L{window_size}"
    return results["models"][model_key]


model_lr_p40 = get_model_from_results(
    results_lr_balanced,
    model_name="logistic_regression",
    target="t2_p40_h30",
    window_size=30,
)

model_gru_p40 = get_model_from_results(
    results_gru_baseline,
    model_name="gru",
    target="t2_p40_h30",
    window_size=30,
)

model_xgb_p40 = get_model_from_results(
    results_xgb_conservative,
    model_name="xgboost",
    target="t2_p40_h30",
    window_size=30,
)

## **15.2. Creación de bundle VALID y TEST**

In [90]:
# =========================================================
# Bundles VALID y TEST | target t2_p40_h30
# =========================================================

target = "t2_p40_h30"
window_size = 30

print("=" * 100)
print("BUILD BUNDLES VALID / TEST")
print("=" * 100)
print(f"target      = {target}")
print(f"window_size = {window_size}")

# ---------------------------------------------------------
# 1) Crear bundle completo
# ---------------------------------------------------------

bundles = create_bundles(
    window_size=window_size,
    targets=[target],
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
)

bundle_full = bundles[target]

print("\nBundle completo cargado correctamente.\n")

# ---------------------------------------------------------
# 2) Crear bundles específicos para VALID y TEST
# ---------------------------------------------------------

bundle_valid = {
    "target": target,
    "horizon": bundle_full.get("horizon"),
    "window_size": bundle_full.get("window_size"),
    "valid": bundle_full["valid"],
}

bundle_test = {
    "target": target,
    "horizon": bundle_full.get("horizon"),
    "window_size": bundle_full.get("window_size"),
    "test": bundle_full["test"],
}

# ---------------------------------------------------------
# 3) Extraer VALID y TEST
# ---------------------------------------------------------

X_valid = bundle_valid["valid"]["X"]
y_valid = bundle_valid["valid"]["y"]

X_test = bundle_test["test"]["X"]
y_test = bundle_test["test"]["y"]

# ---------------------------------------------------------
# 4) Mostrar información VALID
# ---------------------------------------------------------

print("=" * 100)
print("VALID DATA")
print("=" * 100)

print(f"X_valid shape: {X_valid.shape}")
print(f"y_valid shape: {y_valid.shape}")
print(f"dtype X_valid: {X_valid.dtype}")
print(f"dtype y_valid: {y_valid.dtype}")

assert X_valid.shape[1] == window_size, "Window size inconsistente en VALID"
assert len(X_valid) == len(y_valid), "X_valid e y_valid desalineados"

classes_valid = sorted(set(y_valid))
print(f"Clases VALID: {classes_valid}")

nan_count_valid = pd.isna(X_valid).sum()
print(f"NaN en X_valid: {nan_count_valid}")

# ---------------------------------------------------------
# 5) Mostrar información TEST
# ---------------------------------------------------------

print("\n" + "=" * 100)
print("TEST DATA")
print("=" * 100)

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"dtype X_test: {X_test.dtype}")
print(f"dtype y_test: {y_test.dtype}")

assert X_test.shape[1] == window_size, "Window size inconsistente en TEST"
assert len(X_test) == len(y_test), "X_test e y_test desalineados"

classes_test = sorted(set(y_test))
print(f"Clases TEST: {classes_test}")

nan_count_test = pd.isna(X_test).sum()
print(f"NaN en X_test: {nan_count_test}")

# ---------------------------------------------------------
# 6) Resumen final
# ---------------------------------------------------------

print("\n" + "=" * 100)
print("BUNDLES VALID / TEST LISTOS PARA INFERENCIA")
print("=" * 100)
print("Objetos creados:")
print("- bundle_full")
print("- bundle_valid")
print("- bundle_test")

2026-04-25 01:37:16,200 | INFO | Loaded: windows_t2_p40_h30_train.npz
2026-04-25 01:37:16,201 | INFO | X shape: (32147, 30, 7) | y shape: (32147,)
2026-04-25 01:37:16,222 | INFO | Loaded: windows_t2_p40_h30_valid.npz
2026-04-25 01:37:16,223 | INFO | X shape: (6882, 30, 7) | y shape: (6882,)
2026-04-25 01:37:16,245 | INFO | Loaded: windows_t2_p40_h30_test.npz
2026-04-25 01:37:16,246 | INFO | X shape: (6913, 30, 7) | y shape: (6913,)
2026-04-25 01:37:16,251 | INFO | Scaler cargado: scaler_mnq_t2.pkl
2026-04-25 01:37:16,252 | INFO | Bundle cargado | target=t2_p40_h30 | window_size=30 | train=(32147, 30, 7) | valid=(6882, 30, 7) | test=(6913, 30, 7)


BUILD BUNDLES VALID / TEST
target      = t2_p40_h30
window_size = 30

WINDOW_SIZE: 30

TARGET: t2_p40_h30
Train : (32147, 30, 7) (32147,)
Valid : (6882, 30, 7) (6882,)
Test  : (6913, 30, 7) (6913,)
Scaler: StandardScaler

Bundle completo cargado correctamente.

VALID DATA
X_valid shape: (6882, 30, 7)
y_valid shape: (6882,)
dtype X_valid: float32
dtype y_valid: int8
Clases VALID: [np.int8(-1), np.int8(0), np.int8(1)]
NaN en X_valid: 0

TEST DATA
X_test shape: (6913, 30, 7)
y_test shape: (6913,)
dtype X_test: float32
dtype y_test: int8
Clases TEST: [np.int8(-1), np.int8(0), np.int8(1)]
NaN en X_test: 0

BUNDLES VALID / TEST LISTOS PARA INFERENCIA
Objetos creados:
- bundle_full
- bundle_valid
- bundle_test


## **15.3. Inferencia con modelos**

In [91]:
import numpy as np
import pandas as pd
import torch


def predict_model_on_split(
    model,
    bundle,
    *,
    model_type: str,
    split: str = "test",
    input_mode: str = "2d_flat",
    class_labels: list[int] = [-1, 0, 1],
    device: str | None = None,
    verbose: bool = True,
):
    """
    Ejecuta inferencia para modelos ya entrenados en cualquier split:
    - Logistic Regression
    - XGBoost
    - GRU

    split: "valid" o "test"
    """

    if split not in {"valid", "test"}:
        raise ValueError("split debe ser 'valid' o 'test'")

    model_type = model_type.lower()

    # =========================
    # 1) Extraer DATA
    # =========================
    X_split = bundle[split]["X"]
    y_split = bundle[split]["y"]

    target = bundle.get("target", None)
    window_size = bundle.get("window_size", None)
    horizon = bundle.get("horizon", None)

    if verbose:
        print("\n" + "=" * 80)
        print(f"INFERENCIA EN {split.upper()}")
        print("=" * 80)
        print(f"model_type  = {model_type}")
        print(f"target      = {target}")
        print(f"window_size = {window_size}")
        print(f"horizon     = {horizon}")
        print(f"X_{split}      = {X_split.shape}")
        print(f"y_{split}      = {y_split.shape}")

    # =========================
    # 2) LR / XGB
    # =========================
    if model_type in ["lr", "logistic", "logistic_regression", "xgb", "xgboost"]:

        if verbose:
            print("\n[1/4] Preparando input tabular...")
            print(f"input_mode = {input_mode}")

        X_model = prepare_X_for_model(X_split, input_mode=input_mode)

        if verbose:
            print(f"X_model = {X_model.shape}")
            print("[2/4] Ejecutando predict y predict_proba...")

        y_pred = model.predict(X_model)
        y_proba = model.predict_proba(X_model)

        if hasattr(model, "classes_"):
            model_classes = list(model.classes_)
        else:
            model_classes = class_labels

        # XGB codificado
        if model_classes == [0, 1, 2]:
            if verbose:
                print("[INFO] Decodificando clases XGB [0,1,2] -> [-1,0,1]")

            idx_to_class = {0: -1, 1: 0, 2: 1}
            y_pred = np.array([idx_to_class[int(y)] for y in y_pred])
            model_classes = class_labels

    # =========================
    # 3) GRU
    # =========================
    elif model_type == "gru":

        if device is None:
            device = "cuda" if torch.cuda.is_available() else "cpu"

        if verbose:
            print("\n[1/4] Preparando input secuencial para GRU...")
            print(f"device = {device}")

        model = model.to(device)
        model.eval()

        X_t = torch.tensor(X_split, dtype=torch.float32).to(device)

        if verbose:
            print(f"X_tensor = {tuple(X_t.shape)}")
            print("[2/4] Forward pass + softmax...")

        with torch.no_grad():
            logits = model(X_t)
            y_proba = torch.softmax(logits, dim=1).cpu().numpy()

        y_pred_idx = y_proba.argmax(axis=1)
        idx_to_class = {0: -1, 1: 0, 2: 1}
        y_pred = np.array([idx_to_class[int(i)] for i in y_pred_idx])

        model_classes = class_labels

    else:
        raise ValueError("model_type inválido")

    # =========================
    # 4) Validación
    # =========================
    if verbose:
        print("[3/4] Validando clases...")
        print(f"model_classes = {model_classes}")

    if list(model_classes) != class_labels:
        raise ValueError("Orden de clases inconsistente")

    # =========================
    # 5) DataFrame salida
    # =========================
    if verbose:
        print("[4/4] Construyendo DataFrame...")

    df_pred = pd.DataFrame({
        "y_true": y_split,
        "y_pred": y_pred,
        "proba_-1": y_proba[:, 0],
        "proba_0": y_proba[:, 1],
        "proba_1": y_proba[:, 2],
    })

    df_pred["confidence"] = y_proba.max(axis=1)
    df_pred["is_correct"] = df_pred["y_true"].eq(df_pred["y_pred"])

    # =========================
    # 6) Resumen
    # =========================
    if verbose:
        print("\n" + "-" * 80)
        print(f"RESUMEN {split.upper()}")
        print("-" * 80)
        print(f"n_samples       = {len(df_pred)}")
        print(f"accuracy_simple = {df_pred['is_correct'].mean():.6f}")

        print("\nDistribución y_true:")
        print(df_pred["y_true"].value_counts(normalize=True).sort_index())

        print("\nDistribución y_pred:")
        print(df_pred["y_pred"].value_counts(normalize=True).sort_index())

        print("=" * 80)

    return {
        "y_true": y_split,
        "y_pred": y_pred,
        "y_proba": y_proba,
        "df_pred": df_pred,
    }

### **Obtener predicciones por modelo**

In [98]:
# VALID
pred_lr_valid = predict_model_on_split(
    model=model_lr_p40,
    bundle=bundle_full,
    model_type="logistic_regression",
    split="valid",
    verbose=False,
)

# TEST
pred_lr_test = predict_model_on_split(
    model=model_lr_p40,
    bundle=bundle_full,
    model_type="logistic_regression",
    split="test",
    verbose=False,
)

In [99]:
# =========================================================
# Inferencia GRU | VALID y TEST
# =========================================================

pred_gru_valid = predict_model_on_split(
    model=model_gru_p40,
    bundle=bundle_full,
    model_type="gru",
    split="valid",
    verbose=False,
)

pred_gru_test = predict_model_on_split(
    model=model_gru_p40,
    bundle=bundle_full,
    model_type="gru",
    split="test",
    verbose=False,
)




In [100]:
# =========================================================
# Inferencia XGBoost | VALID y TEST
# =========================================================

pred_xgb_valid = predict_model_on_split(
    model=model_xgb_p40,
    bundle=bundle_full,
    model_type="xgboost",
    split="valid",
    verbose=False,
)

pred_xgb_test = predict_model_on_split(
    model=model_xgb_p40,
    bundle=bundle_full,
    model_type="xgboost",
    split="test",
    verbose=False,
)

### **Obtener df_pred**

In [101]:
# =========================================================
# Extraer DataFrames de predicciones
# =========================================================

# VALID
df_lr_p40_valid  = pred_lr_valid["df_pred"]
df_gru_p40_valid = pred_gru_valid["df_pred"]
df_xgb_p40_valid = pred_xgb_valid["df_pred"]

# TEST
df_lr_p40_test  = pred_lr_test["df_pred"]
df_gru_p40_test = pred_gru_test["df_pred"]
df_xgb_p40_test = pred_xgb_test["df_pred"]

## **15.4. Evaluación de predicciones en VALID y TEST**

### **Función de evaluación**

In [103]:
# =========================================================
# Evaluación de predicciones generadas | VALID y TEST
# =========================================================

# ---------------------------------------------------------
# 1) Configuración de evaluaciones
# ---------------------------------------------------------

prediction_evals = [
    {
        "df_pred": df_lr_p40_valid,
        "model_name": "logistic_regression",
        "experiment_name": "lr_final_balanced",
        "split": "valid",
        "threshold_long": 0.40,
        "threshold_short": 0.45,
    },
    {
        "df_pred": df_lr_p40_test,
        "model_name": "logistic_regression",
        "experiment_name": "lr_final_balanced",
        "split": "test",
        "threshold_long": 0.40,
        "threshold_short": 0.45,
    },
    {
        "df_pred": df_gru_p40_valid,
        "model_name": "gru",
        "experiment_name": "gru_final_baseline",
        "split": "valid",
        "threshold_long": 0.40,
        "threshold_short": 0.40,
    },
    {
        "df_pred": df_gru_p40_test,
        "model_name": "gru",
        "experiment_name": "gru_final_baseline",
        "split": "test",
        "threshold_long": 0.40,
        "threshold_short": 0.40,
    },
    {
        "df_pred": df_xgb_p40_valid,
        "model_name": "xgboost",
        "experiment_name": "xgb_final_conservative",
        "split": "valid",
        "threshold_long": 0.50,
        "threshold_short": 0.50,
    },
    {
        "df_pred": df_xgb_p40_test,
        "model_name": "xgboost",
        "experiment_name": "xgb_final_conservative",
        "split": "test",
        "threshold_long": 0.50,
        "threshold_short": 0.50,
    },
]

# ---------------------------------------------------------
# 2) Ejecutar evaluaciones
# ---------------------------------------------------------

eval_results = {}

for cfg in prediction_evals:
    key = f"{cfg['experiment_name']}__{cfg['split']}"

    print("=" * 80)
    print(f"EVALUANDO: {key}")
    print("=" * 80)

    eval_results[key] = evaluate_test_predictions_df(
        df_pred=cfg["df_pred"],
        model_name=cfg["model_name"],
        experiment_name=cfg["experiment_name"],
        target="t2_p40_h30",
        window_size=30,
        horizon=30,
        threshold_long=cfg["threshold_long"],
        threshold_short=cfg["threshold_short"],
        split=cfg["split"],
    )

# ---------------------------------------------------------
# 3) Consolidar métricas
# ---------------------------------------------------------

df_metrics_oos = pd.concat(
    [res["metrics"] for res in eval_results.values()],
    ignore_index=True,
)

# ---------------------------------------------------------
# 4) Consolidar probabilidades
# ---------------------------------------------------------

df_probabilities_oos = pd.concat(
    [res["probabilities"] for res in eval_results.values()],
    ignore_index=True,
)

# ---------------------------------------------------------
# 5) Ordenar resultados
# ---------------------------------------------------------

sort_cols_metrics = [
    c for c in [
        "experiment_name",
        "model",
        "split",
        "target",
        "window_size",
        "horizon",
    ]
    if c in df_metrics_oos.columns
]

df_metrics_oos = (
    df_metrics_oos
    .sort_values(sort_cols_metrics)
    .reset_index(drop=True)
)

sort_cols_prob = [
    c for c in [
        "experiment_name",
        "model",
        "split",
        "target",
        "window_size",
        "horizon",
    ]
    if c in df_probabilities_oos.columns
]

df_probabilities_oos = (
    df_probabilities_oos
    .sort_values(sort_cols_prob)
    .reset_index(drop=True)
)

# ---------------------------------------------------------
# 6) Mostrar resumen
# ---------------------------------------------------------

print("\n" + "=" * 80)
print("MÉTRICAS VALID / TEST")
print("=" * 80)

display_cols = [
    c for c in [
        "experiment_name",
        "model",
        "split",
        "target",
        "threshold_long",
        "threshold_short",
        "accuracy",
        "balanced_accuracy",
        "f1_macro",
        "n_trades",
        "trade_rate",
        "n_useful",
        "precision_useful",
        "useful_rate_total",
    ]
    if c in df_metrics_oos.columns
]

display(df_metrics_oos[display_cols])

print("\n" + "=" * 80)
print("PROBABILIDADES VALID / TEST")
print("=" * 80)

display(df_probabilities_oos.head())

# ---------------------------------------------------------
# 7) Guardar resultados
# ---------------------------------------------------------

df_metrics_oos.to_parquet(
    OUTPUT_DIR_BEST_MODEL / "df_metrics_oos_valid_test.parquet",
    index=False,
)

df_probabilities_oos.to_parquet(
    OUTPUT_DIR_BEST_MODEL / "df_probabilities_oos_valid_test.parquet",
    index=False,
)

print("\nResultados VALID / TEST guardados correctamente.")

EVALUANDO: lr_final_balanced__valid
EVALUANDO: lr_final_balanced__test
EVALUANDO: gru_final_baseline__valid
EVALUANDO: gru_final_baseline__test
EVALUANDO: xgb_final_conservative__valid
EVALUANDO: xgb_final_conservative__test

MÉTRICAS VALID / TEST


,experiment_name,model,split,target,threshold_long,threshold_short,accuracy,balanced_accuracy,f1_macro,n_trades,trade_rate,n_useful,precision_useful,useful_rate_total
0,gru_final_baseline,gru,test,t2_p40_h30,0.4,0.40,0.376248,0.406167,0.352812,916,0.132504,423,0.461790,0.061189
1,gru_final_baseline,gru,valid,t2_p40_h30,0.4,0.40,0.393200,0.420773,0.380090,1691,0.245713,686,0.405677,0.099680
2,lr_final_balanced,logistic_regression,test,t2_p40_h30,0.4,0.45,0.371474,0.396597,0.351934,1110,0.160567,498,0.448649,0.072038
3,lr_final_balanced,logistic_regression,valid,t2_p40_h30,0.4,0.45,0.382302,0.406574,0.373965,1037,0.150683,459,0.442623,0.066696
4,xgb_final_conservative,xgboost,test,t2_p40_h30,0.5,0.50,0.368436,0.388991,0.336305,137,0.019818,48,0.350365,0.006943
5,xgb_final_conservative,xgboost,valid,t2_p40_h30,0.5,0.50,0.384481,0.408318,0.369286,175,0.025429,81,0.462857,0.011770



PROBABILIDADES VALID / TEST


,proba_-1,proba_0,proba_1,pred_label,confidence,y_true,is_correct,signal_raw,trade,experiment_name,model,split,window_size,target,horizon,threshold_long,threshold_short,class_labels
0,0.344057,0.310762,0.345181,1,0.345181,1,True,0,False,gru_final_baseline,gru,test,30,t2_p40_h30,30,0.4,0.4,"[-1, 0, 1]"
1,0.336107,0.319338,0.344555,1,0.344555,1,True,0,False,gru_final_baseline,gru,test,30,t2_p40_h30,30,0.4,0.4,"[-1, 0, 1]"
2,0.333800,0.322083,0.344118,1,0.344118,1,True,0,False,gru_final_baseline,gru,test,30,t2_p40_h30,30,0.4,0.4,"[-1, 0, 1]"
3,0.329694,0.323989,0.346317,1,0.346317,1,True,0,False,gru_final_baseline,gru,test,30,t2_p40_h30,30,0.4,0.4,"[-1, 0, 1]"
4,0.327304,0.324461,0.348236,1,0.348236,1,True,0,False,gru_final_baseline,gru,test,30,t2_p40_h30,30,0.4,0.4,"[-1, 0, 1]"



Resultados VALID / TEST guardados correctamente.


### **Conclusiones del análisis VALID vs TEST**

El modelo GRU baseline se mantiene como el mejor desde el punto de vista predictivo. En TEST conserva el mayor valor de balanced_accuracy (0.4062), por encima de Logistic Regression (0.3966) y XGBoost (0.3890). Si bien se observa una caída respecto a VALID, esta es moderada y no indica sobreajuste severo, lo que confirma una adecuada capacidad de generalización.

El modelo Logistic Regression en su configuración balanced se consolida como el más estable desde el punto de vista operativo. En TEST presenta una leve mejora en sus métricas clave respecto a VALID: precision_useful aumenta de 0.4426 a 0.4486, el trade_rate de 0.1507 a 0.1606 y el useful_rate_total de 0.0667 a 0.0720. Este comportamiento indica que el modelo no solo generaliza correctamente, sino que incluso mejora su desempeño fuera de muestra en términos operativos.

El modelo GRU presenta la mayor precision_useful en TEST (0.4618), lo que lo posiciona como el modelo de mayor calidad de señal. Sin embargo, esta mejora viene acompañada de una reducción significativa en la frecuencia de operación, con un trade_rate que cae de 0.2457 en VALID a 0.1325 en TEST. Esto refleja un comportamiento más selectivo fuera de muestra.

El modelo XGBoost en su configuración conservative pierde solidez en TEST. Su precision_useful cae de 0.4629 a 0.3504 y mantiene un nivel de actividad muy bajo (trade_rate de 0.0198), lo que reduce significativamente su utilidad práctica. En este contexto, deja de ser competitivo frente a los otros modelos.




# **Conclusión final**



El análisis conjunto confirma que no existe sobreajuste severo y que el pipeline de modelado es consistente. Se identifican dos modelos claramente dominantes bajo distintos criterios:

* GRU baseline como modelo principal desde el punto de vista predictivo y de calidad de señal
* Logistic Regression balanced como modelo más estable y eficiente operativamente

En contraste, XGBoost conservative queda descartado como candidato final en esta configuración.

A partir de estos resultados, la etapa siguiente debe centrarse en la definición de estrategias operativas basadas en la combinación de GRU baseline y Logistic Regression balanced, evaluando esquemas como uso individual, intersección de señales y enfoques jerárquicos.